# Day 4 Code-Along: ship it (serve, control, observe, evaluate)

**Hands-on AI Workshop · Day 4 of 4 · Capstone: Research Paper Summarizer Agent with Memory**

Today the agent becomes a service two hundred colleagues could use at once. Cell numbers (4.2, 5.4 …) match
the tags on the slides.

| Section | What you build | Failure shown on purpose |
|---|---|---|
| **4. Serve it** | FastAPI: upload, ask, stream, usage; API-key auth; TODO 1: validate uploads | A text file pretending to be a PDF; an 11 MB upload |
| **5. Control** | LiteLLM Router: alias, retries, fallback; per-user budgets (TODO 2) and rate limits | Broken primary model; Bob runs out of budget; Alice hits the rate limit |
| **6. Observe** | One JSON log line per request; an OpenTelemetry trace tree | A slow vector search, found from the trace |
| **7. Evaluate** | Golden set, rule scorers, LLM judge, baseline.json; TODO 3: the regression gate | A friendlier prompt silently drops citations and is blocked |

**Fell behind, or your runtime restarted?** Run the **⏩ catch-up cell** (5.0, 6.0, 7.0) at the start of the
section you want to rejoin. **A cell failed?** Post the error in the Q&A channel; a TA will help.

### Your provider: Colab Secrets (🔑 in the left sidebar, notebook access ON)

| Secret | What it is |
|---|---|
| `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL` | The same as Days 1-3 (pinned in the channel). The model must support **tool calling** |
| `EMBED_MODEL` | The embedding model, as on Days 2-3 |
| `LLM_FALLBACK_MODEL` | *Optional.* A second model for 5.2; without it the fallback is your primary again |
| `PRICE_IN_PER_M`, `PRICE_OUT_PER_M` | *Optional.* USD per 1M tokens; leave them out for the workshop prices ($0.15 / $0.60) |

> ⚠️ **Never put company data into free APIs.**
>
> 📄 **TinyCoder, its authors and its numbers are FICTIONAL**, written for this workshop. Cell 4.0 draws it as a
> three-page PDF.

> **Production vs Colab, honestly:** in production LiteLLM runs as a **proxy** with Postgres and Redis, and owns
> budgets and rate limits for every app. Colab can't run Postgres for 200 people, so here LiteLLM runs as a
> **library** inside the service, and budgets and limits are a few lines of our own code. Same mechanism: check
> before every call, record after.

### 0.1 Install

Installs today's libraries (pinned: `requirements-day4.txt`). If it asks you to **restart the session**, do it (Runtime → Restart session), then run 4.0.

In [ ]:
import importlib, importlib.metadata, json, os, shutil, sys, time, urllib.request, warnings
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'requirements-day4.txt': 'paper-summarizer/requirements-day4.txt', 'paper_agent/__init__.py': 'paper-summarizer/paper_agent/__init__.py', 'paper_agent/rag/__init__.py': 'paper-summarizer/paper_agent/rag/__init__.py', 'paper_agent/rag/pdf_rag.py': 'paper-summarizer/paper_agent/rag/pdf_rag.py', 'paper_agent/agent/__init__.py': 'paper-summarizer/paper_agent/agent/__init__.py', 'paper_agent/agent/papers.py': 'paper-summarizer/paper_agent/agent/papers.py', 'paper_agent/agent/tools.py': 'paper-summarizer/paper_agent/agent/tools.py', 'paper_agent/agent/guards.py': 'paper-summarizer/paper_agent/agent/guards.py', 'paper_agent/agent/memory.py': 'paper-summarizer/paper_agent/agent/memory.py', 'paper_agent/agent/trace.py': 'paper-summarizer/paper_agent/agent/trace.py', 'paper_agent/agent/model.py': 'paper-summarizer/paper_agent/agent/model.py', 'paper_agent/agent/policy.py': 'paper-summarizer/paper_agent/agent/policy.py', 'paper_agent/agent/fakes.py': 'paper-summarizer/paper_agent/agent/fakes.py', 'paper_agent/prompts/__init__.py': 'paper-summarizer/paper_agent/prompts/__init__.py', 'paper_agent/prompts/agent_v1.txt': 'paper-summarizer/paper_agent/prompts/agent_v1.txt', 'paper_agent/prompts/agent_v2.txt': 'paper-summarizer/paper_agent/prompts/agent_v2.txt', 'paper_agent/prompts/judge_v1.txt': 'paper-summarizer/paper_agent/prompts/judge_v1.txt', 'paper_agent/fixtures/__init__.py': 'paper-summarizer/paper_agent/fixtures/__init__.py', 'paper_agent/fixtures/tinycoder_pdf.py': 'paper-summarizer/paper_agent/fixtures/tinycoder_pdf.py', 'paper_agent/evals/__init__.py': 'paper-summarizer/paper_agent/evals/__init__.py', 'paper_agent/evals/metrics.py': 'paper-summarizer/paper_agent/evals/metrics.py', 'paper_agent/evals/runner.py': 'paper-summarizer/paper_agent/evals/runner.py', 'paper_agent/evals/gate.py': 'paper-summarizer/paper_agent/evals/gate.py', 'paper_agent/evals/golden_tinycoder.json': 'paper-summarizer/paper_agent/evals/golden_tinycoder.json', 'paper_agent/service/__init__.py': 'paper-summarizer/paper_agent/service/__init__.py', 'paper_agent/service/config.py': 'paper-summarizer/paper_agent/service/config.py', 'paper_agent/service/errors.py': 'paper-summarizer/paper_agent/service/errors.py', 'paper_agent/service/gateway.py': 'paper-summarizer/paper_agent/service/gateway.py', 'paper_agent/service/usage.py': 'paper-summarizer/paper_agent/service/usage.py', 'paper_agent/service/store.py': 'paper-summarizer/paper_agent/service/store.py', 'paper_agent/service/pipeline.py': 'paper-summarizer/paper_agent/service/pipeline.py', 'paper_agent/service/validate.py': 'paper-summarizer/paper_agent/service/validate.py', 'paper_agent/service/app.py': 'paper-summarizer/paper_agent/service/app.py', 'paper_agent/service/tracing.py': 'paper-summarizer/paper_agent/service/tracing.py', 'paper_agent/service/keys.py': 'paper-summarizer/paper_agent/service/keys.py'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)
# 0.1 INSTALL: today's libraries, pinned (requirements-day4.txt). Run it once, before 2:00 if you can.
fetch("requirements-day4.txt", force=True)
PINS = dict(line.split("==") for line in Path("requirements-day4.txt").read_text().split() if "==" in line)

def installed(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

wrong = {p: installed(p) for p, v in PINS.items() if installed(p) != v}
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if wrong and not IN_COLAB:                                    # never pip-install into someone's own Python
    raise RuntimeError(f"Not in Colab, and these differ from requirements-day4.txt: {wrong}. "
                       "Install them in this environment yourself (pip install -r requirements-day4.txt).")
if wrong:
    print(f"Installing {len(wrong)} pinned package(s): a minute or two the first time ...")
    !pip install -q -r requirements-day4.txt
    loaded = [m for m in ("pydantic", "fastapi", "httpx", "openai", "langchain_core", "opentelemetry") if m in sys.modules]
    if loaded:
        print("\n" + "=" * 72 + "\n✅ Installed. ⚠️  Now RESTART THE SESSION (Runtime → Restart session), then run 4.0.\n"
              f"Older versions of {', '.join(loaded)} were already loaded in this runtime.\n" + "=" * 72)
        raise SystemExit("restart the session, then run 4.0")
print("✅ today's libraries are installed: run 4.0")

---
# Section 4: Serve it

FastAPI serves the agent from inside this notebook: uvicorn runs in a background thread, and we talk to it over HTTP like any other client would.

### 4.0 Setup and catch-up: Days 1-3 in one cell

The TinyCoder paper as a real PDF, parsing, chunks, retrieval, the LangChain agent and the citation check. It is also the catch-up cell for Section 4.

In [ ]:
# 4.0 SETUP and CATCH-UP: Days 1-3 in one cell. Downloads the course code, connects to your provider through
# LiteLLM, draws the TinyCoder PDF, parses, chunks, embeds and indexes it, and builds the agent with its tools
# and citation check. Safe to run again at any time. You don't need to read it today: you wrote all of it.
import importlib, importlib.metadata, json, os, shutil, sys, time, urllib.request, warnings
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'requirements-day4.txt': 'paper-summarizer/requirements-day4.txt', 'paper_agent/__init__.py': 'paper-summarizer/paper_agent/__init__.py', 'paper_agent/rag/__init__.py': 'paper-summarizer/paper_agent/rag/__init__.py', 'paper_agent/rag/pdf_rag.py': 'paper-summarizer/paper_agent/rag/pdf_rag.py', 'paper_agent/agent/__init__.py': 'paper-summarizer/paper_agent/agent/__init__.py', 'paper_agent/agent/papers.py': 'paper-summarizer/paper_agent/agent/papers.py', 'paper_agent/agent/tools.py': 'paper-summarizer/paper_agent/agent/tools.py', 'paper_agent/agent/guards.py': 'paper-summarizer/paper_agent/agent/guards.py', 'paper_agent/agent/memory.py': 'paper-summarizer/paper_agent/agent/memory.py', 'paper_agent/agent/trace.py': 'paper-summarizer/paper_agent/agent/trace.py', 'paper_agent/agent/model.py': 'paper-summarizer/paper_agent/agent/model.py', 'paper_agent/agent/policy.py': 'paper-summarizer/paper_agent/agent/policy.py', 'paper_agent/agent/fakes.py': 'paper-summarizer/paper_agent/agent/fakes.py', 'paper_agent/prompts/__init__.py': 'paper-summarizer/paper_agent/prompts/__init__.py', 'paper_agent/prompts/agent_v1.txt': 'paper-summarizer/paper_agent/prompts/agent_v1.txt', 'paper_agent/prompts/agent_v2.txt': 'paper-summarizer/paper_agent/prompts/agent_v2.txt', 'paper_agent/prompts/judge_v1.txt': 'paper-summarizer/paper_agent/prompts/judge_v1.txt', 'paper_agent/fixtures/__init__.py': 'paper-summarizer/paper_agent/fixtures/__init__.py', 'paper_agent/fixtures/tinycoder_pdf.py': 'paper-summarizer/paper_agent/fixtures/tinycoder_pdf.py', 'paper_agent/evals/__init__.py': 'paper-summarizer/paper_agent/evals/__init__.py', 'paper_agent/evals/metrics.py': 'paper-summarizer/paper_agent/evals/metrics.py', 'paper_agent/evals/runner.py': 'paper-summarizer/paper_agent/evals/runner.py', 'paper_agent/evals/gate.py': 'paper-summarizer/paper_agent/evals/gate.py', 'paper_agent/evals/golden_tinycoder.json': 'paper-summarizer/paper_agent/evals/golden_tinycoder.json', 'paper_agent/service/__init__.py': 'paper-summarizer/paper_agent/service/__init__.py', 'paper_agent/service/config.py': 'paper-summarizer/paper_agent/service/config.py', 'paper_agent/service/errors.py': 'paper-summarizer/paper_agent/service/errors.py', 'paper_agent/service/gateway.py': 'paper-summarizer/paper_agent/service/gateway.py', 'paper_agent/service/usage.py': 'paper-summarizer/paper_agent/service/usage.py', 'paper_agent/service/store.py': 'paper-summarizer/paper_agent/service/store.py', 'paper_agent/service/pipeline.py': 'paper-summarizer/paper_agent/service/pipeline.py', 'paper_agent/service/validate.py': 'paper-summarizer/paper_agent/service/validate.py', 'paper_agent/service/app.py': 'paper-summarizer/paper_agent/service/app.py', 'paper_agent/service/tracing.py': 'paper-summarizer/paper_agent/service/tracing.py', 'paper_agent/service/keys.py': 'paper-summarizer/paper_agent/service/keys.py'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)
for name in FILES:
    fetch(name)
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")      # LiteLLM: no price download at import
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import logging
for noisy in ("pypdf", "chromadb", "LiteLLM", "LiteLLM Router", "httpx", "uvicorn.error"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# Your provider, from Colab Secrets (🔑, notebook access ON) or environment variables. Table at the top.
# ⚠️ Never put company data into free APIs.
from paper_agent.service import config
config.load_secrets()
missing = [n for n in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL") if not os.environ.get(n)]
assert not missing, f"Missing {missing}: add them in Colab Secrets (🔑) and turn on notebook access, then run 4.0 again."

import queue, re, threading, uuid
from typing import Callable, Optional
import httpx
from fastapi import Depends, FastAPI, File, Header, HTTPException, Request, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.testclient import TestClient
from langchain.agents.middleware import before_model
from langchain_core.messages import AIMessage
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from paper_agent.agent.fakes import FakeToolModel                        # offline model for the TODO tests
from paper_agent.agent.guards import CITATION
from paper_agent.agent.papers import PAPERS
from paper_agent.fixtures.tinycoder_pdf import make_pdf                  # Day 1's paper as a real 3-page PDF
from paper_agent.prompts import load_prompt
from paper_agent.service.config import ALIAS, DEFAULT_BUDGET_USD, MAX_TOKENS, MAX_UPLOAD_MB
from paper_agent.service.errors import BudgetExceededError, RateLimitedError
from paper_agent.service.gateway import CURRENT_KEY, make_chat_model, make_router
from paper_agent.service.pipeline import EXPLAIN_Q, Ctx, ask_paper, build_agent, explain_events
from paper_agent.service.store import PaperStore, setup_retrieval
import paper_agent.service.store as paper_store                          # search(): the `retrieve` span (6.3)
from paper_agent.service.tracing import TRACER, setup_tracing, show_trace
from paper_agent.service.usage import (BUDGETS, CALLS, SPEND, message_cost, record_usage, trace_model_call,
                                       trace_tool_call, usage_of)
from paper_agent.service.app import (API_KEYS, ERROR_LOG, KEY_USERS, LOG, SERVERS, Question, LogCapture,
                                     attach_log_handler, check_rate_limit, error_body, extra_middleware, key_info,
                                     set_rate_limit)
from paper_agent.evals.runner import EVAL_PAUSE_S, baseline_record, chunk_index, golden_set
from paper_agent.evals.gate import compare

# Days 1-3, rebuilt: the gateway (LiteLLM Router), retrieval (Day 2) and the agent (Day 3)
router = make_router()                       # alias paper-explainer -> your model; fallback, retries, timeout (5.1)
setup_retrieval(router, ".")                 # Day 2's embeddings go through the gateway too (alias paper-embed)
PDF_BYTES = make_pdf("tinycoder.pdf")        # FICTIONAL paper, drawn with reportlab: same bytes every time
papers = PaperStore(".")
PAPER_ID, _ = papers.ingest(PDF_BYTES)       # validate, parse, chunk, embed, index (Days 2-3)
agent = build_agent(router=router)           # Day 3's agent: search_paper + get_section, guards, citation check
ALICE, BOB = {"Authorization": "Bearer key-alice"}, {"Authorization": "Bearer key-bob"}
Q_SPEED = "How much faster is TinyCoder than the 7B baseline?"
print(f"✅ paper {PAPER_ID}: {len(papers.get(PAPER_ID)['chunks'])} chunks, indexed · model alias {ALIAS} → "
      f"{os.environ['LLM_MODEL'].split(',')[0]} · agent ready")

### 4.1 Setup check

One question through the agent: the answer, its citations, how many LLM calls it took, the tokens and the cost. When you see **ready**, put a green tick in the chat.

In [ ]:
# 4.1
result = ask_paper(PAPER_ID, Q_SPEED, "setup-check", agent)
print("answer   :", result["answer"])
print("citations:", result["citations"], "valid" if result["citations_valid"] else "NOT VALID")
print(f"LLM calls: {result['llm_calls']} · tokens: {result['input_tokens']:,} in + {result['output_tokens']:,} out"
      f" · cost: ${result['cost_usd']:.6f} · latency: {result['latency_ms']:,} ms")
print("✅ ready" if result["citations_valid"] and result["llm_calls"] else "⚠️ something is off: post this output in the channel")

### 4.2 The service

The whole service. Four endpoints: `POST /papers` (upload), `POST /papers/{id}/ask`, `POST /papers/{id}/explain`
(streamed) and `GET /me/usage`. `current_user` maps every API key to a user (at work: your single sign-on).
`error_response` turns our errors into clear HTTP answers. The `request_log` middleware writes one line per
request: silent for now, switched on in Section 6.

In [ ]:
def current_user(request: Request, authorization: Optional[str] = Header(None)) -> str:
    key = (authorization or "").removeprefix("Bearer ").strip()
    if config.GATEWAY_MODE == "proxy" and key:
        if key not in KEY_USERS:
            KEY_USERS[key] = key_info(key).get("user_id") or "unknown"
        user = KEY_USERS[key]
    else:
        user = API_KEYS.get(key)
    if not user:
        raise HTTPException(401, "Unknown API key.")
    request.state.user, request.state.api_key = user, key if config.GATEWAY_MODE == "proxy" else None
    check_rate_limit(user)
    return user


def error_response(exc: BaseException) -> tuple:
    """(status, JSONResponse) for an exception that escaped a route."""
    import litellm
    import openai
    if isinstance(exc, BudgetExceededError):
        return 429, JSONResponse(error_body("budget_exceeded", (
            f"You've used your budget for today (${exc.spent:.6f} of ${exc.budget:.6f}). It resets at midnight "
            "UTC; ask your admin if you need more.")), status_code=429)
    if isinstance(exc, RateLimitedError):
        headers = {"Retry-After": str(exc.retry_after)} if exc.retry_after else {}
        wait = f"Try again in {exc.retry_after} seconds." if exc.retry_after else "Try again in a minute."
        limit = f" ({exc.limit} per minute)" if exc.limit else ""
        return 429, JSONResponse(error_body("rate_limited", f"Too many requests{limit}. {wait}"),
                                 status_code=429, headers=headers)
    if isinstance(exc, (litellm.exceptions.APIError, litellm.exceptions.ServiceUnavailableError,
                        litellm.exceptions.APIConnectionError, litellm.exceptions.Timeout,
                        litellm.exceptions.InternalServerError, openai.APIConnectionError, openai.InternalServerError)):
        ERROR_LOG.warning("provider error %s: %s", type(exc).__name__, str(exc)[:200])
        return 503, JSONResponse(error_body("provider_unavailable", "The model provider is unavailable right now. "
                                                                    "Try again in a minute."), status_code=503)
    ERROR_LOG.error("unhandled %s: %s", type(exc).__name__, str(exc)[:300])
    return 500, JSONResponse(error_body("internal_error", "Something went wrong on our side."), status_code=500)


def build_app(agent=None, store: Optional[PaperStore] = None) -> FastAPI:
    quiet = {"/health"}                    # health checks every few seconds: no spans, no log lines
    app = FastAPI(title="Paper explainer", version="4.0",
                  telemetry={"exclude": lambda scope: scope.get("path") in quiet})
    app.state.agents = {}
    app.state.agent = agent
    app.state.store = store or PaperStore(os.environ.get("DATA_DIR", "."))

    def agent_for(version: Optional[str]):
        if version is None or version == getattr(app.state.agent, "prompt_version", None):
            if app.state.agent is None:
                app.state.agent = build_agent()
            return app.state.agent
        if version not in ("v1", "v2"):
            raise HTTPException(422, "prompt_version must be v1 or v2.")
        if version not in app.state.agents:
            app.state.agents[version] = build_agent(version, extra_middleware=extra_middleware())
        return app.state.agents[version]

    def paper_or_404(paper_id: str) -> None:
        if not app.state.store.get(paper_id):
            raise HTTPException(404, f"No paper {paper_id}: upload it first (POST /papers).")

    @app.get("/health")
    def health():
        return {"status": "ok", "gateway_mode": config.GATEWAY_MODE}

    @app.post("/papers")
    def upload(request: Request, file: UploadFile = File(...), user: str = Depends(current_user)):  # noqa: B008
        data = file.file.read(config.MAX_UPLOAD_MB * 1024 * 1024 + 1)    # never read more than the limit + 1
        validate_upload(data)
        CURRENT_KEY.set(request.state.api_key)          # proxy mode: the uploader's key pays for the embeddings
        paper_id, is_new = app.state.store.ingest(data)
        chunks = len(app.state.store.get(paper_id)["chunks"])
        return {"paper_id": paper_id, "filename": file.filename, "chunks": chunks,
                "status": "processed" if is_new else "already processed"}

    @app.post("/papers/{paper_id}/ask")
    def ask(paper_id: str, body: Question, request: Request, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        result = ask_paper(paper_id, body.question, user, agent=agent_for(body.prompt_version),
                           api_key=request.state.api_key)
        request.state.usage = result
        return {"paper_id": paper_id, **result}

    @app.post("/papers/{paper_id}/explain")
    def explain(paper_id: str, request: Request, body: Optional[Question] = None, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        events: queue.Queue = queue.Queue()
        agent = agent_for(body.prompt_version if body else None)

        def run():               # one thread for the whole run, so its spans open and close in one context
            try:
                for event, data in explain_events(paper_id, body.question if body else EXPLAIN_Q, user, agent,
                                                  request.state.api_key):
                    if event == "done":
                        request.state.usage = data
                    events.put((event, data))
            except Exception as exc:                       # mid-stream: the status is already 200
                status, payload = error_response(exc)
                events.put(("error", {**json.loads(payload.body), "status": status}))
            events.put(None)

        threading.Thread(target=run, daemon=True).start()

        def stream():
            while (item := events.get()) is not None:
                yield f"event: {item[0]}\ndata: {json.dumps(item[1])}\n\n"
        return StreamingResponse(stream(), media_type="text/event-stream")

    @app.get("/me/usage")
    def my_usage(request: Request, user: str = Depends(current_user)):
        if config.GATEWAY_MODE == "proxy":
            info = key_info(request.state.api_key)
            spent, budget = info.get("spend") or 0.0, info.get("max_budget")
            return {"user": user, "llm_calls": None, "spent_usd": round(spent, 6), "budget_usd": budget,
                    "remaining_usd": None if budget is None else round(budget - spent, 6), "source": "gateway"}
        return usage_of(user)

    # --- errors: a status and a sentence, never a stack trace ----------------------------------------
    @app.exception_handler(BudgetExceededError)
    @app.exception_handler(RateLimitedError)
    def too_much(request, exc):
        return error_response(exc)[1]

    @app.exception_handler(HTTPException)
    def http_error(request, exc: HTTPException):
        slug = {401: "unauthorized", 404: "not_found", 413: "too_large", 415: "unsupported_type",
                422: "invalid_request", 503: "gateway_unavailable"}.get(exc.status_code, "error")
        return JSONResponse(error_body(slug, exc.detail), status_code=exc.status_code)

    @app.middleware("http")
    async def request_log(request: Request, call_next):
        """One JSON line per request, written when the response has been sent. The question, the answer and
        the paper's text are never logged."""
        request_id = uuid.uuid4().hex[:8]
        started = time.perf_counter()
        try:
            response = await call_next(request)
        except Exception as exc:                            # anything not handled above
            status, response = error_response(exc)
        response.headers["X-Request-ID"] = request_id
        body = response.body_iterator if hasattr(response, "body_iterator") else None

        def write():
            usage = getattr(request.state, "usage", None) or {}
            LOG.info(json.dumps({
                "request_id": request_id, "user": getattr(request.state, "user", None),
                "route": f"{request.method} {request.url.path}", "status": response.status_code,
                "latency_ms": round((time.perf_counter() - started) * 1000),
                "llm_calls": usage.get("llm_calls", 0), "input_tokens": usage.get("input_tokens", 0),
                "output_tokens": usage.get("output_tokens", 0), "cost_usd": usage.get("cost_usd", 0.0),
                "answered_by": usage.get("answered_by", [])}))

        if request.url.path in quiet:
            return response
        if body is None:
            write()
            return response

        async def logged():
            async for chunk in body:
                yield chunk
            write()
        response.body_iterator = logged()
        return response

    return app


app = build_app(agent, papers)

In [ ]:
# 4.2: the four routes
for route in app.routes:
    if route.path.startswith(("/papers", "/me")):
        print(f"{', '.join(sorted(route.methods)):5} {route.path}")

### 4.3 Your turn: never trust the file. ✏️ TODO 1 (two lines)

The filename and the `Content-Type` both come from the client. The bytes don't lie: every PDF starts with
`%PDF-`. **TODO 1:** (1) if `data` doesn't start with `b"%PDF-"`, raise `HTTPException(415, ...)`; (2) if it is
bigger than `MAX_UPLOAD_MB`, raise `HTTPException(413, ...)`. Check before parsing: a huge or broken file is
exactly the work we want to avoid.

In [ ]:
def validate_upload(data: bytes) -> None:
    """415 unless the bytes start like a PDF; 413 if they are over MAX_UPLOAD_MB."""
    # ✏️ TODO 1 (two lines):
    #   1. not starting with b"%PDF-"  -> raise HTTPException(415, "Only PDF files are accepted.")
    #   2. len(data) > MAX_UPLOAD_MB * 1024 * 1024 -> raise HTTPException(413, ...)
    pass

### 4.3a Test: offline, no server and no LLM call

A text file named `.pdf`, an 11 MB upload and a wrong API key. You want **415, 413 and 401**.

In [ ]:
# 4.3a test
test_client = TestClient(app, raise_server_exceptions=False)
fake_pdf = test_client.post("/papers", headers=ALICE, files={"file": ("notes.pdf", b"Just some notes, not a PDF.", "application/pdf")})
too_big = test_client.post("/papers", headers=ALICE, files={"file": ("huge.pdf", b"%PDF-1.4\n" + b"0" * (11 * 1024 * 1024), "application/pdf")})
wrong_key = test_client.post("/papers", headers={"Authorization": "Bearer key-mallory"},
                             files={"file": ("tinycoder.pdf", PDF_BYTES, "application/pdf")})
for label, r in (("text file named .pdf", fake_pdf), ("11 MB upload", too_big), ("wrong API key", wrong_key)):
    print(f"{label:22} → {r.status_code} {r.json()}")
assert fake_pdf.status_code == 415, "TODO 1: check the first bytes: data.startswith(b'%PDF-'), else HTTPException(415, ...)"
assert too_big.status_code == 413, "TODO 1: check the size: len(data) > MAX_UPLOAD_MB * 1024 * 1024 -> HTTPException(413, ...)"
assert wrong_key.status_code == 401
print("\n✅ TODO 1 passed: 415, 413 and 401, before any parsing")

<details><summary>🆘 Stuck? Hint</summary>

Two lines, in this order:

```
if not data.startswith(b"%PDF-"): raise HTTPException(415, "Only PDF files are accepted.")
if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise HTTPException(413, f"Max upload size is {MAX_UPLOAD_MB} MB.")
```
</details>

🆘 **Rescue cell:** tests failing, or you fell behind? Run the next cell, then carry on.

In [ ]:
# 🆘 RESCUE: the finished version. Running it is always safe.
def validate_upload(data: bytes) -> None:
    """415 unless the bytes start like a PDF; 413 if they are over MAX_UPLOAD_MB."""
    if not data.startswith(b"%PDF-"): raise HTTPException(415, "Only PDF files are accepted.")
    if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise HTTPException(413, f"Max upload size is {MAX_UPLOAD_MB} MB.")

### 4.4 Start the server and upload a paper

The server runs in the background of this notebook (port 8765), and we talk to it over HTTP. The same file
twice, under two names: the paper ID is a hash of the **content**, not the name, so it is parsed and embedded
once, whoever uploads it. (4.0 already processed it, so both say so.)

In [ ]:
def start_server(app: FastAPI, port: int = 8765) -> str:
    """4.4: serve `app` with uvicorn in a background thread of this notebook. Running it again stops the old
    server first, so the cell (and every catch-up cell) can be re-run."""
    import uvicorn
    if port in SERVERS:
        old, thread = SERVERS.pop(port)
        old.should_exit = True
        thread.join(timeout=10)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        if not thread.is_alive():
            raise RuntimeError(f"the server could not start: is port {port} in use? Runtime → Restart session, "
                               "then run 5.0")
        time.sleep(0.05)
    SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


BASE = start_server(app)
http = httpx.Client(base_url=BASE, timeout=120, trust_env=False)

In [ ]:
# 4.4
for name in ("tinycoder.pdf", "tinycoder_FINAL_v2.pdf"):
    r = http.post("/papers", headers=ALICE, files={"file": (name, PDF_BYTES, "application/pdf")})
    print(f"{name:24} → {r.status_code} {r.json()['paper_id']}  {r.json()['status']}")

### 4.5 Ask a question over HTTP

JSON in, JSON out. Besides the answer: citations and whether they are valid, LLM calls, tokens, cost, and which model answered. And in the headers, a request ID: keep it for Section 6.

In [ ]:
# 4.5
r = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": Q_SPEED})
body = r.json()
print("X-Request-ID:", r.headers["X-Request-ID"], "| status", r.status_code)
print(json.dumps({k: body[k] for k in ("answer", "citations", "citations_valid", "llm_calls", "input_tokens",
                                       "output_tokens", "cost_usd", "answered_by")}, indent=1))
LAST_COST = body["cost_usd"]                       # what one question costs: 5.4 uses it

### 4.6 Streaming progress

Server-sent events: a plain HTTP response that stays open and sends lines as they are ready. Compare the two numbers at the bottom.

In [ ]:
# 4.6
start, first, full = time.perf_counter(), None, None
with http.stream("POST", f"/papers/{PAPER_ID}/explain", headers=ALICE) as r:
    event = None
    for line in r.iter_lines():
        if line.startswith("event: "):
            event = line[7:]
        elif line.startswith("data: "):
            t = time.perf_counter() - start
            first = first or t
            data = json.loads(line[6:])
            if event == "answer":
                full = t
            shown = {"progress": lambda d: " ".join(str(v) for k, v in d.items()),
                     "answer": lambda d: d["answer"][:90] + " …",
                     "done": lambda d: f"{d['llm_calls']} LLM calls, ${d['cost_usd']:.6f}"}.get(event, str)(data)
            print(f"{t:5.1f}s  {event:9} {shown}")
print(f"\nfirst event after {first:.1f}s · full answer after {full:.1f}s")

---
# Section 5: Control cost and failures

A gateway in front of every model call, a budget per user checked before every call, and a rate limit per user.

In [ ]:
# ⏩ 5.0 CATCH-UP: run ONLY if you joined late or your runtime restarted.
# It recreates everything earlier sections defined, so you can follow along from here.
# 4.0 SETUP and CATCH-UP: Days 1-3 in one cell. Downloads the course code, connects to your provider through
# LiteLLM, draws the TinyCoder PDF, parses, chunks, embeds and indexes it, and builds the agent with its tools
# and citation check. Safe to run again at any time. You don't need to read it today: you wrote all of it.
import importlib, importlib.metadata, json, os, shutil, sys, time, urllib.request, warnings
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'requirements-day4.txt': 'paper-summarizer/requirements-day4.txt', 'paper_agent/__init__.py': 'paper-summarizer/paper_agent/__init__.py', 'paper_agent/rag/__init__.py': 'paper-summarizer/paper_agent/rag/__init__.py', 'paper_agent/rag/pdf_rag.py': 'paper-summarizer/paper_agent/rag/pdf_rag.py', 'paper_agent/agent/__init__.py': 'paper-summarizer/paper_agent/agent/__init__.py', 'paper_agent/agent/papers.py': 'paper-summarizer/paper_agent/agent/papers.py', 'paper_agent/agent/tools.py': 'paper-summarizer/paper_agent/agent/tools.py', 'paper_agent/agent/guards.py': 'paper-summarizer/paper_agent/agent/guards.py', 'paper_agent/agent/memory.py': 'paper-summarizer/paper_agent/agent/memory.py', 'paper_agent/agent/trace.py': 'paper-summarizer/paper_agent/agent/trace.py', 'paper_agent/agent/model.py': 'paper-summarizer/paper_agent/agent/model.py', 'paper_agent/agent/policy.py': 'paper-summarizer/paper_agent/agent/policy.py', 'paper_agent/agent/fakes.py': 'paper-summarizer/paper_agent/agent/fakes.py', 'paper_agent/prompts/__init__.py': 'paper-summarizer/paper_agent/prompts/__init__.py', 'paper_agent/prompts/agent_v1.txt': 'paper-summarizer/paper_agent/prompts/agent_v1.txt', 'paper_agent/prompts/agent_v2.txt': 'paper-summarizer/paper_agent/prompts/agent_v2.txt', 'paper_agent/prompts/judge_v1.txt': 'paper-summarizer/paper_agent/prompts/judge_v1.txt', 'paper_agent/fixtures/__init__.py': 'paper-summarizer/paper_agent/fixtures/__init__.py', 'paper_agent/fixtures/tinycoder_pdf.py': 'paper-summarizer/paper_agent/fixtures/tinycoder_pdf.py', 'paper_agent/evals/__init__.py': 'paper-summarizer/paper_agent/evals/__init__.py', 'paper_agent/evals/metrics.py': 'paper-summarizer/paper_agent/evals/metrics.py', 'paper_agent/evals/runner.py': 'paper-summarizer/paper_agent/evals/runner.py', 'paper_agent/evals/gate.py': 'paper-summarizer/paper_agent/evals/gate.py', 'paper_agent/evals/golden_tinycoder.json': 'paper-summarizer/paper_agent/evals/golden_tinycoder.json', 'paper_agent/service/__init__.py': 'paper-summarizer/paper_agent/service/__init__.py', 'paper_agent/service/config.py': 'paper-summarizer/paper_agent/service/config.py', 'paper_agent/service/errors.py': 'paper-summarizer/paper_agent/service/errors.py', 'paper_agent/service/gateway.py': 'paper-summarizer/paper_agent/service/gateway.py', 'paper_agent/service/usage.py': 'paper-summarizer/paper_agent/service/usage.py', 'paper_agent/service/store.py': 'paper-summarizer/paper_agent/service/store.py', 'paper_agent/service/pipeline.py': 'paper-summarizer/paper_agent/service/pipeline.py', 'paper_agent/service/validate.py': 'paper-summarizer/paper_agent/service/validate.py', 'paper_agent/service/app.py': 'paper-summarizer/paper_agent/service/app.py', 'paper_agent/service/tracing.py': 'paper-summarizer/paper_agent/service/tracing.py', 'paper_agent/service/keys.py': 'paper-summarizer/paper_agent/service/keys.py'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)
for name in FILES:
    fetch(name)
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")      # LiteLLM: no price download at import
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import logging
for noisy in ("pypdf", "chromadb", "LiteLLM", "LiteLLM Router", "httpx", "uvicorn.error"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# Your provider, from Colab Secrets (🔑, notebook access ON) or environment variables. Table at the top.
# ⚠️ Never put company data into free APIs.
from paper_agent.service import config
config.load_secrets()
missing = [n for n in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL") if not os.environ.get(n)]
assert not missing, f"Missing {missing}: add them in Colab Secrets (🔑) and turn on notebook access, then run 4.0 again."

import queue, re, threading, uuid
from typing import Callable, Optional
import httpx
from fastapi import Depends, FastAPI, File, Header, HTTPException, Request, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.testclient import TestClient
from langchain.agents.middleware import before_model
from langchain_core.messages import AIMessage
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from paper_agent.agent.fakes import FakeToolModel                        # offline model for the TODO tests
from paper_agent.agent.guards import CITATION
from paper_agent.agent.papers import PAPERS
from paper_agent.fixtures.tinycoder_pdf import make_pdf                  # Day 1's paper as a real 3-page PDF
from paper_agent.prompts import load_prompt
from paper_agent.service.config import ALIAS, DEFAULT_BUDGET_USD, MAX_TOKENS, MAX_UPLOAD_MB
from paper_agent.service.errors import BudgetExceededError, RateLimitedError
from paper_agent.service.gateway import CURRENT_KEY, make_chat_model, make_router
from paper_agent.service.pipeline import EXPLAIN_Q, Ctx, ask_paper, build_agent, explain_events
from paper_agent.service.store import PaperStore, setup_retrieval
import paper_agent.service.store as paper_store                          # search(): the `retrieve` span (6.3)
from paper_agent.service.tracing import TRACER, setup_tracing, show_trace
from paper_agent.service.usage import (BUDGETS, CALLS, SPEND, message_cost, record_usage, trace_model_call,
                                       trace_tool_call, usage_of)
from paper_agent.service.app import (API_KEYS, ERROR_LOG, KEY_USERS, LOG, SERVERS, Question, LogCapture,
                                     attach_log_handler, check_rate_limit, error_body, extra_middleware, key_info,
                                     set_rate_limit)
from paper_agent.evals.runner import EVAL_PAUSE_S, baseline_record, chunk_index, golden_set
from paper_agent.evals.gate import compare

# Days 1-3, rebuilt: the gateway (LiteLLM Router), retrieval (Day 2) and the agent (Day 3)
router = make_router()                       # alias paper-explainer -> your model; fallback, retries, timeout (5.1)
setup_retrieval(router, ".")                 # Day 2's embeddings go through the gateway too (alias paper-embed)
PDF_BYTES = make_pdf("tinycoder.pdf")        # FICTIONAL paper, drawn with reportlab: same bytes every time
papers = PaperStore(".")
PAPER_ID, _ = papers.ingest(PDF_BYTES)       # validate, parse, chunk, embed, index (Days 2-3)
agent = build_agent(router=router)           # Day 3's agent: search_paper + get_section, guards, citation check
ALICE, BOB = {"Authorization": "Bearer key-alice"}, {"Authorization": "Bearer key-bob"}
Q_SPEED = "How much faster is TinyCoder than the 7B baseline?"
print(f"✅ paper {PAPER_ID}: {len(papers.get(PAPER_ID)['chunks'])} chunks, indexed · model alias {ALIAS} → "
      f"{os.environ['LLM_MODEL'].split(',')[0]} · agent ready")

def current_user(request: Request, authorization: Optional[str] = Header(None)) -> str:
    key = (authorization or "").removeprefix("Bearer ").strip()
    if config.GATEWAY_MODE == "proxy" and key:
        if key not in KEY_USERS:
            KEY_USERS[key] = key_info(key).get("user_id") or "unknown"
        user = KEY_USERS[key]
    else:
        user = API_KEYS.get(key)
    if not user:
        raise HTTPException(401, "Unknown API key.")
    request.state.user, request.state.api_key = user, key if config.GATEWAY_MODE == "proxy" else None
    check_rate_limit(user)
    return user


def error_response(exc: BaseException) -> tuple:
    """(status, JSONResponse) for an exception that escaped a route."""
    import litellm
    import openai
    if isinstance(exc, BudgetExceededError):
        return 429, JSONResponse(error_body("budget_exceeded", (
            f"You've used your budget for today (${exc.spent:.6f} of ${exc.budget:.6f}). It resets at midnight "
            "UTC; ask your admin if you need more.")), status_code=429)
    if isinstance(exc, RateLimitedError):
        headers = {"Retry-After": str(exc.retry_after)} if exc.retry_after else {}
        wait = f"Try again in {exc.retry_after} seconds." if exc.retry_after else "Try again in a minute."
        limit = f" ({exc.limit} per minute)" if exc.limit else ""
        return 429, JSONResponse(error_body("rate_limited", f"Too many requests{limit}. {wait}"),
                                 status_code=429, headers=headers)
    if isinstance(exc, (litellm.exceptions.APIError, litellm.exceptions.ServiceUnavailableError,
                        litellm.exceptions.APIConnectionError, litellm.exceptions.Timeout,
                        litellm.exceptions.InternalServerError, openai.APIConnectionError, openai.InternalServerError)):
        ERROR_LOG.warning("provider error %s: %s", type(exc).__name__, str(exc)[:200])
        return 503, JSONResponse(error_body("provider_unavailable", "The model provider is unavailable right now. "
                                                                    "Try again in a minute."), status_code=503)
    ERROR_LOG.error("unhandled %s: %s", type(exc).__name__, str(exc)[:300])
    return 500, JSONResponse(error_body("internal_error", "Something went wrong on our side."), status_code=500)


def build_app(agent=None, store: Optional[PaperStore] = None) -> FastAPI:
    quiet = {"/health"}                    # health checks every few seconds: no spans, no log lines
    app = FastAPI(title="Paper explainer", version="4.0",
                  telemetry={"exclude": lambda scope: scope.get("path") in quiet})
    app.state.agents = {}
    app.state.agent = agent
    app.state.store = store or PaperStore(os.environ.get("DATA_DIR", "."))

    def agent_for(version: Optional[str]):
        if version is None or version == getattr(app.state.agent, "prompt_version", None):
            if app.state.agent is None:
                app.state.agent = build_agent()
            return app.state.agent
        if version not in ("v1", "v2"):
            raise HTTPException(422, "prompt_version must be v1 or v2.")
        if version not in app.state.agents:
            app.state.agents[version] = build_agent(version, extra_middleware=extra_middleware())
        return app.state.agents[version]

    def paper_or_404(paper_id: str) -> None:
        if not app.state.store.get(paper_id):
            raise HTTPException(404, f"No paper {paper_id}: upload it first (POST /papers).")

    @app.get("/health")
    def health():
        return {"status": "ok", "gateway_mode": config.GATEWAY_MODE}

    @app.post("/papers")
    def upload(request: Request, file: UploadFile = File(...), user: str = Depends(current_user)):  # noqa: B008
        data = file.file.read(config.MAX_UPLOAD_MB * 1024 * 1024 + 1)    # never read more than the limit + 1
        validate_upload(data)
        CURRENT_KEY.set(request.state.api_key)          # proxy mode: the uploader's key pays for the embeddings
        paper_id, is_new = app.state.store.ingest(data)
        chunks = len(app.state.store.get(paper_id)["chunks"])
        return {"paper_id": paper_id, "filename": file.filename, "chunks": chunks,
                "status": "processed" if is_new else "already processed"}

    @app.post("/papers/{paper_id}/ask")
    def ask(paper_id: str, body: Question, request: Request, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        result = ask_paper(paper_id, body.question, user, agent=agent_for(body.prompt_version),
                           api_key=request.state.api_key)
        request.state.usage = result
        return {"paper_id": paper_id, **result}

    @app.post("/papers/{paper_id}/explain")
    def explain(paper_id: str, request: Request, body: Optional[Question] = None, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        events: queue.Queue = queue.Queue()
        agent = agent_for(body.prompt_version if body else None)

        def run():               # one thread for the whole run, so its spans open and close in one context
            try:
                for event, data in explain_events(paper_id, body.question if body else EXPLAIN_Q, user, agent,
                                                  request.state.api_key):
                    if event == "done":
                        request.state.usage = data
                    events.put((event, data))
            except Exception as exc:                       # mid-stream: the status is already 200
                status, payload = error_response(exc)
                events.put(("error", {**json.loads(payload.body), "status": status}))
            events.put(None)

        threading.Thread(target=run, daemon=True).start()

        def stream():
            while (item := events.get()) is not None:
                yield f"event: {item[0]}\ndata: {json.dumps(item[1])}\n\n"
        return StreamingResponse(stream(), media_type="text/event-stream")

    @app.get("/me/usage")
    def my_usage(request: Request, user: str = Depends(current_user)):
        if config.GATEWAY_MODE == "proxy":
            info = key_info(request.state.api_key)
            spent, budget = info.get("spend") or 0.0, info.get("max_budget")
            return {"user": user, "llm_calls": None, "spent_usd": round(spent, 6), "budget_usd": budget,
                    "remaining_usd": None if budget is None else round(budget - spent, 6), "source": "gateway"}
        return usage_of(user)

    # --- errors: a status and a sentence, never a stack trace ----------------------------------------
    @app.exception_handler(BudgetExceededError)
    @app.exception_handler(RateLimitedError)
    def too_much(request, exc):
        return error_response(exc)[1]

    @app.exception_handler(HTTPException)
    def http_error(request, exc: HTTPException):
        slug = {401: "unauthorized", 404: "not_found", 413: "too_large", 415: "unsupported_type",
                422: "invalid_request", 503: "gateway_unavailable"}.get(exc.status_code, "error")
        return JSONResponse(error_body(slug, exc.detail), status_code=exc.status_code)

    @app.middleware("http")
    async def request_log(request: Request, call_next):
        """One JSON line per request, written when the response has been sent. The question, the answer and
        the paper's text are never logged."""
        request_id = uuid.uuid4().hex[:8]
        started = time.perf_counter()
        try:
            response = await call_next(request)
        except Exception as exc:                            # anything not handled above
            status, response = error_response(exc)
        response.headers["X-Request-ID"] = request_id
        body = response.body_iterator if hasattr(response, "body_iterator") else None

        def write():
            usage = getattr(request.state, "usage", None) or {}
            LOG.info(json.dumps({
                "request_id": request_id, "user": getattr(request.state, "user", None),
                "route": f"{request.method} {request.url.path}", "status": response.status_code,
                "latency_ms": round((time.perf_counter() - started) * 1000),
                "llm_calls": usage.get("llm_calls", 0), "input_tokens": usage.get("input_tokens", 0),
                "output_tokens": usage.get("output_tokens", 0), "cost_usd": usage.get("cost_usd", 0.0),
                "answered_by": usage.get("answered_by", [])}))

        if request.url.path in quiet:
            return response
        if body is None:
            write()
            return response

        async def logged():
            async for chunk in body:
                yield chunk
            write()
        response.body_iterator = logged()
        return response

    return app


app = build_app(agent, papers)

def validate_upload(data: bytes) -> None:
    """415 unless the bytes start like a PDF; 413 if they are over MAX_UPLOAD_MB."""
    if not data.startswith(b"%PDF-"): raise HTTPException(415, "Only PDF files are accepted.")
    if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise HTTPException(413, f"Max upload size is {MAX_UPLOAD_MB} MB.")

def start_server(app: FastAPI, port: int = 8765) -> str:
    """4.4: serve `app` with uvicorn in a background thread of this notebook. Running it again stops the old
    server first, so the cell (and every catch-up cell) can be re-run."""
    import uvicorn
    if port in SERVERS:
        old, thread = SERVERS.pop(port)
        old.should_exit = True
        thread.join(timeout=10)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        if not thread.is_alive():
            raise RuntimeError(f"the server could not start: is port {port} in use? Runtime → Restart session, "
                               "then run 5.0")
        time.sleep(0.05)
    SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


BASE = start_server(app)
http = httpx.Client(base_url=BASE, timeout=120, trust_env=False)
print('✅ caught up: ready for Section 5')

### 5.1 Open up the gateway

You've been using LiteLLM since 4.0. Our code only ever asks for the alias `paper-explainer`; the Router maps it to a real model, and knows the fallback, the retries and the timeout.

In [ ]:
# 5.1: the Router behind every model call (keys hidden)
for d in router.model_list:
    p = d["litellm_params"]
    print(f"{d['model_name']:26} → {p['model']:34} (deployment id: {d['model_info']['id']})")
print("\nfallbacks  :", router.fallbacks)
print("num_retries:", router.num_retries, "| timeout:", router.timeout, "s")
chat = make_chat_model(router)
print(f"LangChain  : {type(chat).__name__}(model_name={chat.model_name!r}, max_tokens={chat.max_tokens}, "
      f"max_retries={chat.max_retries})  ← one attempt, no retries")

| Day 1 `llm_client.py` (by hand) | Day 4 (one home each) |
|---|---|
| Retries + backoff | LiteLLM `num_retries` |
| Timeout | LiteLLM `timeout` |
| Fallback model | LiteLLM `fallbacks` |
| `max_tokens` | `max_tokens` on every model call (`MAX_TOKENS = 400`) |
| Budget guard | per user, before every model call (5.3) |
| Log line | one per request, plus a trace (6.x) |

LangChain's `max_retries=1` means **one attempt**: one retry layer, the Day 1 rule.

### 5.2 Force a fallback

Point the primary at a model that doesn't exist and ask through the server. No restart: the next request just uses the new agent. The cell puts the real primary back at the end.

In [ ]:
# 5.2
app.state.agent = build_agent(router=make_router(primary="this-model-does-not-exist"))
r = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": Q_SPEED})
print("status     :", r.status_code)
print("answered_by:", r.json().get("answered_by"), "← the primary failed, the fallback answered")
print("answer     :", r.json().get("answer", r.json())[:100], "…")
app.state.agent = agent                              # the real primary again
print("\nprimary restored:", http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE,
                                        json={"question": Q_SPEED}).json()["answered_by"])

### 5.3 Your turn: a budget on every call. ✏️ TODO 2 (two lines)

Check once per question and one question can still run four model calls straight past the budget. So
`budget_guard` is a `@before_model` middleware: LangChain runs it before **every** model call. `record_usage`
(written in 4.0) adds each call's cost to `SPEND` after it. **TODO 2:** inside `check_budget`, if the user has
spent their whole budget, raise `BudgetExceededError(user_id, spent, budget)`.

In [ ]:
def check_budget(user_id: str) -> None:
    """Raise BudgetExceededError once the user has spent their whole budget."""
    spent = SPEND[user_id]
    budget = BUDGETS.get(user_id, DEFAULT_BUDGET_USD)
    # ✏️ TODO 2 (two lines): if spent has reached the budget,
    #   raise BudgetExceededError(user_id, spent, budget)
    pass


@before_model
def budget_guard(state, runtime):
    """Before EVERY model call, not once per question: one question can take several calls."""
    if config.GATEWAY_MODE == "proxy":
        return None
    check_budget(runtime.context.user_id)
    return None

### 5.3a Test: offline, with a fake model

Under the budget passes, at or over raises, and an agent run stops **before** its next model call once the spend crosses the budget.

In [ ]:
# 5.3a test
BUDGETS["test-user"], SPEND["test-user"] = 0.001, 0.0009
check_budget("test-user")                                       # under: fine
for spent in (0.001, 0.002):
    SPEND["test-user"] = spent
    try:
        check_budget("test-user")
        raise AssertionError(f"TODO 2: spent ${spent} of a $0.001 budget should raise BudgetExceededError")
    except BudgetExceededError as e:
        print("raised as expected:", e)
fake = FakeToolModel(script=[AIMessage("", tool_calls=[{"name": "get_section", "args": {"section_id": "results"}, "id": "t1"}]),
                             AIMessage("", tool_calls=[{"name": "get_section", "args": {"section_id": "limitations"}, "id": "t2"}]),
                             AIMessage("Done.")])
BUDGETS["test-user"], SPEND["test-user"] = 1e-9, 0.0             # the first call spends more than this
try:
    build_agent(model=fake, extra_middleware=[budget_guard]).invoke(
        {"messages": [{"role": "user", "content": "Tell me everything."}]}, context=Ctx("test-user", PAPER_ID))
except BudgetExceededError:
    pass
print(f"model calls made: {len(fake.requests)} (the script had 3)")
assert len(fake.requests) == 1, "TODO 2: budget_guard should stop the run before the second model call"
print("✅ TODO 2 passed: checked before every model call")

<details><summary>🆘 Stuck? Hint</summary>

Two lines, where the TODO is:

```
if spent >= budget:
    raise BudgetExceededError(user_id, spent, budget)
```
</details>

🆘 **Rescue cell:** tests failing, or you fell behind? Run the next cell, then carry on.

In [ ]:
# 🆘 RESCUE: the finished version. Running it is always safe.
def check_budget(user_id: str) -> None:
    """Raise BudgetExceededError once the user has spent their whole budget."""
    spent = SPEND[user_id]
    budget = BUDGETS.get(user_id, DEFAULT_BUDGET_USD)
    if spent >= budget:
        raise BudgetExceededError(user_id, spent, budget)


@before_model
def budget_guard(state, runtime):
    """Before EVERY model call, not once per question: one question can take several calls."""
    if config.GATEWAY_MODE == "proxy":
        return None
    check_budget(runtime.context.user_id)
    return None

### 5.4 Two users, two budgets

The guard joins the live agent. Alice gets a dollar; Bob gets about one and a half questions' worth, based on what the last question actually cost. Watch Bob, then the last line.

In [ ]:
agent = build_agent(router=router, extra_middleware=[budget_guard])    # 5.4: the guard joins the live agent
app.state.agent = agent
BUDGETS["alice"] = 1.00

In [ ]:
# 5.4
if "LAST_COST" not in globals():                    # joined late: measure one question now
    LAST_COST = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": Q_SPEED}).json()["cost_usd"]
BUDGETS["bob"] = round(1.5 * LAST_COST, 6)
SPEND["bob"], CALLS["bob"] = 0.0, 0
print(f"Bob's budget: ${BUDGETS['bob']:.6f} (1.5 × the last question's ${LAST_COST:.6f})\n")
for n in (1, 2, 3):
    r = http.post(f"/papers/{PAPER_ID}/ask", headers=BOB, json={"question": Q_SPEED})
    print(f"Bob, question {n}: {r.status_code}", r.json()["message"] if r.status_code != 200 else f"(spent so far ${SPEND['bob']:.6f})")
r = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": Q_SPEED})
print(f"Alice           : {r.status_code} (budgets are per user)")
over = SPEND["bob"] - BUDGETS["bob"]
print(f"\nBob spent ${SPEND['bob']:.6f} of ${BUDGETS['bob']:.6f}: over by ${over:.6f}. The check runs before a call, "
      "the cost is known after it, so one call can overshoot. max_tokens on every call keeps that call small.")

### 5.5 Rate limits

Three requests per minute for Alice, then five requests. The cell removes the limit at the end. (In-process counter: fine for one Colab process; the proxy keeps it in Redis.)

In [ ]:
# 5.5
set_rate_limit("alice", 3)
for n in range(1, 6):
    r = http.get("/me/usage", headers=ALICE)
    extra = f"Retry-After: {r.headers['Retry-After']}s · {r.json()['message']}" if r.status_code == 429 else ""
    print(f"request {n}: {r.status_code} {extra}")
set_rate_limit("alice", None)                       # no limit for the rest of the notebook

### 5.6 Spend per user

Each user's calls, spend, budget and what's left. Bob's remaining is negative: the overshoot from 5.4. In production these numbers live in the LiteLLM proxy (slide).

In [ ]:
# 5.6
for user, headers in (("alice", ALICE), ("bob", BOB)):
    print(json.dumps(http.get("/me/usage", headers=headers).json()))

---
# Section 6: Observe it

A log line says what happened in a request; a trace says where the time went.

In [ ]:
# ⏩ 6.0 CATCH-UP: run ONLY if you joined late or your runtime restarted.
# It recreates everything earlier sections defined, so you can follow along from here.
# 4.0 SETUP and CATCH-UP: Days 1-3 in one cell. Downloads the course code, connects to your provider through
# LiteLLM, draws the TinyCoder PDF, parses, chunks, embeds and indexes it, and builds the agent with its tools
# and citation check. Safe to run again at any time. You don't need to read it today: you wrote all of it.
import importlib, importlib.metadata, json, os, shutil, sys, time, urllib.request, warnings
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'requirements-day4.txt': 'paper-summarizer/requirements-day4.txt', 'paper_agent/__init__.py': 'paper-summarizer/paper_agent/__init__.py', 'paper_agent/rag/__init__.py': 'paper-summarizer/paper_agent/rag/__init__.py', 'paper_agent/rag/pdf_rag.py': 'paper-summarizer/paper_agent/rag/pdf_rag.py', 'paper_agent/agent/__init__.py': 'paper-summarizer/paper_agent/agent/__init__.py', 'paper_agent/agent/papers.py': 'paper-summarizer/paper_agent/agent/papers.py', 'paper_agent/agent/tools.py': 'paper-summarizer/paper_agent/agent/tools.py', 'paper_agent/agent/guards.py': 'paper-summarizer/paper_agent/agent/guards.py', 'paper_agent/agent/memory.py': 'paper-summarizer/paper_agent/agent/memory.py', 'paper_agent/agent/trace.py': 'paper-summarizer/paper_agent/agent/trace.py', 'paper_agent/agent/model.py': 'paper-summarizer/paper_agent/agent/model.py', 'paper_agent/agent/policy.py': 'paper-summarizer/paper_agent/agent/policy.py', 'paper_agent/agent/fakes.py': 'paper-summarizer/paper_agent/agent/fakes.py', 'paper_agent/prompts/__init__.py': 'paper-summarizer/paper_agent/prompts/__init__.py', 'paper_agent/prompts/agent_v1.txt': 'paper-summarizer/paper_agent/prompts/agent_v1.txt', 'paper_agent/prompts/agent_v2.txt': 'paper-summarizer/paper_agent/prompts/agent_v2.txt', 'paper_agent/prompts/judge_v1.txt': 'paper-summarizer/paper_agent/prompts/judge_v1.txt', 'paper_agent/fixtures/__init__.py': 'paper-summarizer/paper_agent/fixtures/__init__.py', 'paper_agent/fixtures/tinycoder_pdf.py': 'paper-summarizer/paper_agent/fixtures/tinycoder_pdf.py', 'paper_agent/evals/__init__.py': 'paper-summarizer/paper_agent/evals/__init__.py', 'paper_agent/evals/metrics.py': 'paper-summarizer/paper_agent/evals/metrics.py', 'paper_agent/evals/runner.py': 'paper-summarizer/paper_agent/evals/runner.py', 'paper_agent/evals/gate.py': 'paper-summarizer/paper_agent/evals/gate.py', 'paper_agent/evals/golden_tinycoder.json': 'paper-summarizer/paper_agent/evals/golden_tinycoder.json', 'paper_agent/service/__init__.py': 'paper-summarizer/paper_agent/service/__init__.py', 'paper_agent/service/config.py': 'paper-summarizer/paper_agent/service/config.py', 'paper_agent/service/errors.py': 'paper-summarizer/paper_agent/service/errors.py', 'paper_agent/service/gateway.py': 'paper-summarizer/paper_agent/service/gateway.py', 'paper_agent/service/usage.py': 'paper-summarizer/paper_agent/service/usage.py', 'paper_agent/service/store.py': 'paper-summarizer/paper_agent/service/store.py', 'paper_agent/service/pipeline.py': 'paper-summarizer/paper_agent/service/pipeline.py', 'paper_agent/service/validate.py': 'paper-summarizer/paper_agent/service/validate.py', 'paper_agent/service/app.py': 'paper-summarizer/paper_agent/service/app.py', 'paper_agent/service/tracing.py': 'paper-summarizer/paper_agent/service/tracing.py', 'paper_agent/service/keys.py': 'paper-summarizer/paper_agent/service/keys.py'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)
for name in FILES:
    fetch(name)
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")      # LiteLLM: no price download at import
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import logging
for noisy in ("pypdf", "chromadb", "LiteLLM", "LiteLLM Router", "httpx", "uvicorn.error"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# Your provider, from Colab Secrets (🔑, notebook access ON) or environment variables. Table at the top.
# ⚠️ Never put company data into free APIs.
from paper_agent.service import config
config.load_secrets()
missing = [n for n in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL") if not os.environ.get(n)]
assert not missing, f"Missing {missing}: add them in Colab Secrets (🔑) and turn on notebook access, then run 4.0 again."

import queue, re, threading, uuid
from typing import Callable, Optional
import httpx
from fastapi import Depends, FastAPI, File, Header, HTTPException, Request, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.testclient import TestClient
from langchain.agents.middleware import before_model
from langchain_core.messages import AIMessage
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from paper_agent.agent.fakes import FakeToolModel                        # offline model for the TODO tests
from paper_agent.agent.guards import CITATION
from paper_agent.agent.papers import PAPERS
from paper_agent.fixtures.tinycoder_pdf import make_pdf                  # Day 1's paper as a real 3-page PDF
from paper_agent.prompts import load_prompt
from paper_agent.service.config import ALIAS, DEFAULT_BUDGET_USD, MAX_TOKENS, MAX_UPLOAD_MB
from paper_agent.service.errors import BudgetExceededError, RateLimitedError
from paper_agent.service.gateway import CURRENT_KEY, make_chat_model, make_router
from paper_agent.service.pipeline import EXPLAIN_Q, Ctx, ask_paper, build_agent, explain_events
from paper_agent.service.store import PaperStore, setup_retrieval
import paper_agent.service.store as paper_store                          # search(): the `retrieve` span (6.3)
from paper_agent.service.tracing import TRACER, setup_tracing, show_trace
from paper_agent.service.usage import (BUDGETS, CALLS, SPEND, message_cost, record_usage, trace_model_call,
                                       trace_tool_call, usage_of)
from paper_agent.service.app import (API_KEYS, ERROR_LOG, KEY_USERS, LOG, SERVERS, Question, LogCapture,
                                     attach_log_handler, check_rate_limit, error_body, extra_middleware, key_info,
                                     set_rate_limit)
from paper_agent.evals.runner import EVAL_PAUSE_S, baseline_record, chunk_index, golden_set
from paper_agent.evals.gate import compare

# Days 1-3, rebuilt: the gateway (LiteLLM Router), retrieval (Day 2) and the agent (Day 3)
router = make_router()                       # alias paper-explainer -> your model; fallback, retries, timeout (5.1)
setup_retrieval(router, ".")                 # Day 2's embeddings go through the gateway too (alias paper-embed)
PDF_BYTES = make_pdf("tinycoder.pdf")        # FICTIONAL paper, drawn with reportlab: same bytes every time
papers = PaperStore(".")
PAPER_ID, _ = papers.ingest(PDF_BYTES)       # validate, parse, chunk, embed, index (Days 2-3)
agent = build_agent(router=router)           # Day 3's agent: search_paper + get_section, guards, citation check
ALICE, BOB = {"Authorization": "Bearer key-alice"}, {"Authorization": "Bearer key-bob"}
Q_SPEED = "How much faster is TinyCoder than the 7B baseline?"
print(f"✅ paper {PAPER_ID}: {len(papers.get(PAPER_ID)['chunks'])} chunks, indexed · model alias {ALIAS} → "
      f"{os.environ['LLM_MODEL'].split(',')[0]} · agent ready")

def current_user(request: Request, authorization: Optional[str] = Header(None)) -> str:
    key = (authorization or "").removeprefix("Bearer ").strip()
    if config.GATEWAY_MODE == "proxy" and key:
        if key not in KEY_USERS:
            KEY_USERS[key] = key_info(key).get("user_id") or "unknown"
        user = KEY_USERS[key]
    else:
        user = API_KEYS.get(key)
    if not user:
        raise HTTPException(401, "Unknown API key.")
    request.state.user, request.state.api_key = user, key if config.GATEWAY_MODE == "proxy" else None
    check_rate_limit(user)
    return user


def error_response(exc: BaseException) -> tuple:
    """(status, JSONResponse) for an exception that escaped a route."""
    import litellm
    import openai
    if isinstance(exc, BudgetExceededError):
        return 429, JSONResponse(error_body("budget_exceeded", (
            f"You've used your budget for today (${exc.spent:.6f} of ${exc.budget:.6f}). It resets at midnight "
            "UTC; ask your admin if you need more.")), status_code=429)
    if isinstance(exc, RateLimitedError):
        headers = {"Retry-After": str(exc.retry_after)} if exc.retry_after else {}
        wait = f"Try again in {exc.retry_after} seconds." if exc.retry_after else "Try again in a minute."
        limit = f" ({exc.limit} per minute)" if exc.limit else ""
        return 429, JSONResponse(error_body("rate_limited", f"Too many requests{limit}. {wait}"),
                                 status_code=429, headers=headers)
    if isinstance(exc, (litellm.exceptions.APIError, litellm.exceptions.ServiceUnavailableError,
                        litellm.exceptions.APIConnectionError, litellm.exceptions.Timeout,
                        litellm.exceptions.InternalServerError, openai.APIConnectionError, openai.InternalServerError)):
        ERROR_LOG.warning("provider error %s: %s", type(exc).__name__, str(exc)[:200])
        return 503, JSONResponse(error_body("provider_unavailable", "The model provider is unavailable right now. "
                                                                    "Try again in a minute."), status_code=503)
    ERROR_LOG.error("unhandled %s: %s", type(exc).__name__, str(exc)[:300])
    return 500, JSONResponse(error_body("internal_error", "Something went wrong on our side."), status_code=500)


def build_app(agent=None, store: Optional[PaperStore] = None) -> FastAPI:
    quiet = {"/health"}                    # health checks every few seconds: no spans, no log lines
    app = FastAPI(title="Paper explainer", version="4.0",
                  telemetry={"exclude": lambda scope: scope.get("path") in quiet})
    app.state.agents = {}
    app.state.agent = agent
    app.state.store = store or PaperStore(os.environ.get("DATA_DIR", "."))

    def agent_for(version: Optional[str]):
        if version is None or version == getattr(app.state.agent, "prompt_version", None):
            if app.state.agent is None:
                app.state.agent = build_agent()
            return app.state.agent
        if version not in ("v1", "v2"):
            raise HTTPException(422, "prompt_version must be v1 or v2.")
        if version not in app.state.agents:
            app.state.agents[version] = build_agent(version, extra_middleware=extra_middleware())
        return app.state.agents[version]

    def paper_or_404(paper_id: str) -> None:
        if not app.state.store.get(paper_id):
            raise HTTPException(404, f"No paper {paper_id}: upload it first (POST /papers).")

    @app.get("/health")
    def health():
        return {"status": "ok", "gateway_mode": config.GATEWAY_MODE}

    @app.post("/papers")
    def upload(request: Request, file: UploadFile = File(...), user: str = Depends(current_user)):  # noqa: B008
        data = file.file.read(config.MAX_UPLOAD_MB * 1024 * 1024 + 1)    # never read more than the limit + 1
        validate_upload(data)
        CURRENT_KEY.set(request.state.api_key)          # proxy mode: the uploader's key pays for the embeddings
        paper_id, is_new = app.state.store.ingest(data)
        chunks = len(app.state.store.get(paper_id)["chunks"])
        return {"paper_id": paper_id, "filename": file.filename, "chunks": chunks,
                "status": "processed" if is_new else "already processed"}

    @app.post("/papers/{paper_id}/ask")
    def ask(paper_id: str, body: Question, request: Request, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        result = ask_paper(paper_id, body.question, user, agent=agent_for(body.prompt_version),
                           api_key=request.state.api_key)
        request.state.usage = result
        return {"paper_id": paper_id, **result}

    @app.post("/papers/{paper_id}/explain")
    def explain(paper_id: str, request: Request, body: Optional[Question] = None, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        events: queue.Queue = queue.Queue()
        agent = agent_for(body.prompt_version if body else None)

        def run():               # one thread for the whole run, so its spans open and close in one context
            try:
                for event, data in explain_events(paper_id, body.question if body else EXPLAIN_Q, user, agent,
                                                  request.state.api_key):
                    if event == "done":
                        request.state.usage = data
                    events.put((event, data))
            except Exception as exc:                       # mid-stream: the status is already 200
                status, payload = error_response(exc)
                events.put(("error", {**json.loads(payload.body), "status": status}))
            events.put(None)

        threading.Thread(target=run, daemon=True).start()

        def stream():
            while (item := events.get()) is not None:
                yield f"event: {item[0]}\ndata: {json.dumps(item[1])}\n\n"
        return StreamingResponse(stream(), media_type="text/event-stream")

    @app.get("/me/usage")
    def my_usage(request: Request, user: str = Depends(current_user)):
        if config.GATEWAY_MODE == "proxy":
            info = key_info(request.state.api_key)
            spent, budget = info.get("spend") or 0.0, info.get("max_budget")
            return {"user": user, "llm_calls": None, "spent_usd": round(spent, 6), "budget_usd": budget,
                    "remaining_usd": None if budget is None else round(budget - spent, 6), "source": "gateway"}
        return usage_of(user)

    # --- errors: a status and a sentence, never a stack trace ----------------------------------------
    @app.exception_handler(BudgetExceededError)
    @app.exception_handler(RateLimitedError)
    def too_much(request, exc):
        return error_response(exc)[1]

    @app.exception_handler(HTTPException)
    def http_error(request, exc: HTTPException):
        slug = {401: "unauthorized", 404: "not_found", 413: "too_large", 415: "unsupported_type",
                422: "invalid_request", 503: "gateway_unavailable"}.get(exc.status_code, "error")
        return JSONResponse(error_body(slug, exc.detail), status_code=exc.status_code)

    @app.middleware("http")
    async def request_log(request: Request, call_next):
        """One JSON line per request, written when the response has been sent. The question, the answer and
        the paper's text are never logged."""
        request_id = uuid.uuid4().hex[:8]
        started = time.perf_counter()
        try:
            response = await call_next(request)
        except Exception as exc:                            # anything not handled above
            status, response = error_response(exc)
        response.headers["X-Request-ID"] = request_id
        body = response.body_iterator if hasattr(response, "body_iterator") else None

        def write():
            usage = getattr(request.state, "usage", None) or {}
            LOG.info(json.dumps({
                "request_id": request_id, "user": getattr(request.state, "user", None),
                "route": f"{request.method} {request.url.path}", "status": response.status_code,
                "latency_ms": round((time.perf_counter() - started) * 1000),
                "llm_calls": usage.get("llm_calls", 0), "input_tokens": usage.get("input_tokens", 0),
                "output_tokens": usage.get("output_tokens", 0), "cost_usd": usage.get("cost_usd", 0.0),
                "answered_by": usage.get("answered_by", [])}))

        if request.url.path in quiet:
            return response
        if body is None:
            write()
            return response

        async def logged():
            async for chunk in body:
                yield chunk
            write()
        response.body_iterator = logged()
        return response

    return app


app = build_app(agent, papers)

def validate_upload(data: bytes) -> None:
    """415 unless the bytes start like a PDF; 413 if they are over MAX_UPLOAD_MB."""
    if not data.startswith(b"%PDF-"): raise HTTPException(415, "Only PDF files are accepted.")
    if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise HTTPException(413, f"Max upload size is {MAX_UPLOAD_MB} MB.")

def start_server(app: FastAPI, port: int = 8765) -> str:
    """4.4: serve `app` with uvicorn in a background thread of this notebook. Running it again stops the old
    server first, so the cell (and every catch-up cell) can be re-run."""
    import uvicorn
    if port in SERVERS:
        old, thread = SERVERS.pop(port)
        old.should_exit = True
        thread.join(timeout=10)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        if not thread.is_alive():
            raise RuntimeError(f"the server could not start: is port {port} in use? Runtime → Restart session, "
                               "then run 5.0")
        time.sleep(0.05)
    SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


BASE = start_server(app)
http = httpx.Client(base_url=BASE, timeout=120, trust_env=False)

def check_budget(user_id: str) -> None:
    """Raise BudgetExceededError once the user has spent their whole budget."""
    spent = SPEND[user_id]
    budget = BUDGETS.get(user_id, DEFAULT_BUDGET_USD)
    if spent >= budget:
        raise BudgetExceededError(user_id, spent, budget)


@before_model
def budget_guard(state, runtime):
    """Before EVERY model call, not once per question: one question can take several calls."""
    if config.GATEWAY_MODE == "proxy":
        return None
    check_budget(runtime.context.user_id)
    return None

agent = build_agent(router=router, extra_middleware=[budget_guard])    # 5.4: the guard joins the live agent
app.state.agent = agent
BUDGETS["alice"] = 1.00
print('✅ caught up: ready for Section 6')

### 6.1 One log line per request

The service has written a line for every request since 4.2; nobody was listening. Attach a handler (in production: Datadog, CloudWatch …). JSON, so a machine can search it, and no question, answer or document text in it.

In [ ]:
log = attach_log_handler(LogCapture())      # prints each line, and keeps it for searching

In [ ]:
# 6.1
r = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": "What attention mechanism does TinyCoder use?"})
ticket = r.headers["X-Request-ID"]
print(f"\n🎫 Support ticket: 'this answer was wrong', request id {ticket} (from the response header)")
print("one search →", json.dumps(log.find(ticket)))

### 6.2 Traces

The code loaded in 4.0 already marks each step with a span; they went nowhere. Switch collection on (in memory here; one line changes it to Jaeger or Datadog), ask one question, read the tree.

In [ ]:
exporter = setup_tracing(InMemorySpanExporter())      # from now on, every span is kept here

In [ ]:
# 6.2: the span lines that were in 4.0's code all along
import inspect, paper_agent.service.pipeline, paper_agent.service.usage
for module in (paper_agent.service.pipeline, paper_agent.service.usage, paper_store):
    for line in inspect.getsource(module).splitlines():
        if "start_as_current_span(" in line:
            print(f"{module.__name__.split('.')[-1]:9}│ {line.strip()}")
exporter.clear()
r = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": "What are the main limitations of TinyCoder?"})
print()
spans = show_trace(exporter)

### 6.3 Find the slow step

The vector search gets a second and a half slower, as if the database were overloaded. The log line says the request was slow; the trace says why. The cell resets the delay.

In [ ]:
# 6.3
SEARCH_DELAY_S = 1.5
paper_store.SEARCH_DELAY_S = SEARCH_DELAY_S
exporter.clear()
r = http.post(f"/papers/{PAPER_ID}/ask", headers=ALICE, json={"question": Q_SPEED})
print(f"\n😤 user: 'it took {log.find(r.headers['X-Request-ID'])['latency_ms'] / 1000:.1f} seconds'\n")
spans = show_trace(exporter)
retrieve = [s for s in spans if s.name == "retrieve"]
print(f"\n→ retrieve: {sum((s.end_time - s.start_time) / 1e6 for s in retrieve):,.0f} ms of the request. Not the model.")
paper_store.SEARCH_DELAY_S = SEARCH_DELAY_S = 0.0          # back to normal

---
# Section 7: Evaluate it

A golden set, scorers, a baseline and a gate: unit testing, for prompts.

In [ ]:
# ⏩ 7.0 CATCH-UP: run ONLY if you joined late or your runtime restarted.
# It recreates everything earlier sections defined, so you can follow along from here.
# 4.0 SETUP and CATCH-UP: Days 1-3 in one cell. Downloads the course code, connects to your provider through
# LiteLLM, draws the TinyCoder PDF, parses, chunks, embeds and indexes it, and builds the agent with its tools
# and citation check. Safe to run again at any time. You don't need to read it today: you wrote all of it.
import importlib, importlib.metadata, json, os, shutil, sys, time, urllib.request, warnings
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'requirements-day4.txt': 'paper-summarizer/requirements-day4.txt', 'paper_agent/__init__.py': 'paper-summarizer/paper_agent/__init__.py', 'paper_agent/rag/__init__.py': 'paper-summarizer/paper_agent/rag/__init__.py', 'paper_agent/rag/pdf_rag.py': 'paper-summarizer/paper_agent/rag/pdf_rag.py', 'paper_agent/agent/__init__.py': 'paper-summarizer/paper_agent/agent/__init__.py', 'paper_agent/agent/papers.py': 'paper-summarizer/paper_agent/agent/papers.py', 'paper_agent/agent/tools.py': 'paper-summarizer/paper_agent/agent/tools.py', 'paper_agent/agent/guards.py': 'paper-summarizer/paper_agent/agent/guards.py', 'paper_agent/agent/memory.py': 'paper-summarizer/paper_agent/agent/memory.py', 'paper_agent/agent/trace.py': 'paper-summarizer/paper_agent/agent/trace.py', 'paper_agent/agent/model.py': 'paper-summarizer/paper_agent/agent/model.py', 'paper_agent/agent/policy.py': 'paper-summarizer/paper_agent/agent/policy.py', 'paper_agent/agent/fakes.py': 'paper-summarizer/paper_agent/agent/fakes.py', 'paper_agent/prompts/__init__.py': 'paper-summarizer/paper_agent/prompts/__init__.py', 'paper_agent/prompts/agent_v1.txt': 'paper-summarizer/paper_agent/prompts/agent_v1.txt', 'paper_agent/prompts/agent_v2.txt': 'paper-summarizer/paper_agent/prompts/agent_v2.txt', 'paper_agent/prompts/judge_v1.txt': 'paper-summarizer/paper_agent/prompts/judge_v1.txt', 'paper_agent/fixtures/__init__.py': 'paper-summarizer/paper_agent/fixtures/__init__.py', 'paper_agent/fixtures/tinycoder_pdf.py': 'paper-summarizer/paper_agent/fixtures/tinycoder_pdf.py', 'paper_agent/evals/__init__.py': 'paper-summarizer/paper_agent/evals/__init__.py', 'paper_agent/evals/metrics.py': 'paper-summarizer/paper_agent/evals/metrics.py', 'paper_agent/evals/runner.py': 'paper-summarizer/paper_agent/evals/runner.py', 'paper_agent/evals/gate.py': 'paper-summarizer/paper_agent/evals/gate.py', 'paper_agent/evals/golden_tinycoder.json': 'paper-summarizer/paper_agent/evals/golden_tinycoder.json', 'paper_agent/service/__init__.py': 'paper-summarizer/paper_agent/service/__init__.py', 'paper_agent/service/config.py': 'paper-summarizer/paper_agent/service/config.py', 'paper_agent/service/errors.py': 'paper-summarizer/paper_agent/service/errors.py', 'paper_agent/service/gateway.py': 'paper-summarizer/paper_agent/service/gateway.py', 'paper_agent/service/usage.py': 'paper-summarizer/paper_agent/service/usage.py', 'paper_agent/service/store.py': 'paper-summarizer/paper_agent/service/store.py', 'paper_agent/service/pipeline.py': 'paper-summarizer/paper_agent/service/pipeline.py', 'paper_agent/service/validate.py': 'paper-summarizer/paper_agent/service/validate.py', 'paper_agent/service/app.py': 'paper-summarizer/paper_agent/service/app.py', 'paper_agent/service/tracing.py': 'paper-summarizer/paper_agent/service/tracing.py', 'paper_agent/service/keys.py': 'paper-summarizer/paper_agent/service/keys.py'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)
for name in FILES:
    fetch(name)
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")      # LiteLLM: no price download at import
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import logging
for noisy in ("pypdf", "chromadb", "LiteLLM", "LiteLLM Router", "httpx", "uvicorn.error"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# Your provider, from Colab Secrets (🔑, notebook access ON) or environment variables. Table at the top.
# ⚠️ Never put company data into free APIs.
from paper_agent.service import config
config.load_secrets()
missing = [n for n in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL") if not os.environ.get(n)]
assert not missing, f"Missing {missing}: add them in Colab Secrets (🔑) and turn on notebook access, then run 4.0 again."

import queue, re, threading, uuid
from typing import Callable, Optional
import httpx
from fastapi import Depends, FastAPI, File, Header, HTTPException, Request, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.testclient import TestClient
from langchain.agents.middleware import before_model
from langchain_core.messages import AIMessage
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from paper_agent.agent.fakes import FakeToolModel                        # offline model for the TODO tests
from paper_agent.agent.guards import CITATION
from paper_agent.agent.papers import PAPERS
from paper_agent.fixtures.tinycoder_pdf import make_pdf                  # Day 1's paper as a real 3-page PDF
from paper_agent.prompts import load_prompt
from paper_agent.service.config import ALIAS, DEFAULT_BUDGET_USD, MAX_TOKENS, MAX_UPLOAD_MB
from paper_agent.service.errors import BudgetExceededError, RateLimitedError
from paper_agent.service.gateway import CURRENT_KEY, make_chat_model, make_router
from paper_agent.service.pipeline import EXPLAIN_Q, Ctx, ask_paper, build_agent, explain_events
from paper_agent.service.store import PaperStore, setup_retrieval
import paper_agent.service.store as paper_store                          # search(): the `retrieve` span (6.3)
from paper_agent.service.tracing import TRACER, setup_tracing, show_trace
from paper_agent.service.usage import (BUDGETS, CALLS, SPEND, message_cost, record_usage, trace_model_call,
                                       trace_tool_call, usage_of)
from paper_agent.service.app import (API_KEYS, ERROR_LOG, KEY_USERS, LOG, SERVERS, Question, LogCapture,
                                     attach_log_handler, check_rate_limit, error_body, extra_middleware, key_info,
                                     set_rate_limit)
from paper_agent.evals.runner import EVAL_PAUSE_S, baseline_record, chunk_index, golden_set
from paper_agent.evals.gate import compare

# Days 1-3, rebuilt: the gateway (LiteLLM Router), retrieval (Day 2) and the agent (Day 3)
router = make_router()                       # alias paper-explainer -> your model; fallback, retries, timeout (5.1)
setup_retrieval(router, ".")                 # Day 2's embeddings go through the gateway too (alias paper-embed)
PDF_BYTES = make_pdf("tinycoder.pdf")        # FICTIONAL paper, drawn with reportlab: same bytes every time
papers = PaperStore(".")
PAPER_ID, _ = papers.ingest(PDF_BYTES)       # validate, parse, chunk, embed, index (Days 2-3)
agent = build_agent(router=router)           # Day 3's agent: search_paper + get_section, guards, citation check
ALICE, BOB = {"Authorization": "Bearer key-alice"}, {"Authorization": "Bearer key-bob"}
Q_SPEED = "How much faster is TinyCoder than the 7B baseline?"
print(f"✅ paper {PAPER_ID}: {len(papers.get(PAPER_ID)['chunks'])} chunks, indexed · model alias {ALIAS} → "
      f"{os.environ['LLM_MODEL'].split(',')[0]} · agent ready")

def current_user(request: Request, authorization: Optional[str] = Header(None)) -> str:
    key = (authorization or "").removeprefix("Bearer ").strip()
    if config.GATEWAY_MODE == "proxy" and key:
        if key not in KEY_USERS:
            KEY_USERS[key] = key_info(key).get("user_id") or "unknown"
        user = KEY_USERS[key]
    else:
        user = API_KEYS.get(key)
    if not user:
        raise HTTPException(401, "Unknown API key.")
    request.state.user, request.state.api_key = user, key if config.GATEWAY_MODE == "proxy" else None
    check_rate_limit(user)
    return user


def error_response(exc: BaseException) -> tuple:
    """(status, JSONResponse) for an exception that escaped a route."""
    import litellm
    import openai
    if isinstance(exc, BudgetExceededError):
        return 429, JSONResponse(error_body("budget_exceeded", (
            f"You've used your budget for today (${exc.spent:.6f} of ${exc.budget:.6f}). It resets at midnight "
            "UTC; ask your admin if you need more.")), status_code=429)
    if isinstance(exc, RateLimitedError):
        headers = {"Retry-After": str(exc.retry_after)} if exc.retry_after else {}
        wait = f"Try again in {exc.retry_after} seconds." if exc.retry_after else "Try again in a minute."
        limit = f" ({exc.limit} per minute)" if exc.limit else ""
        return 429, JSONResponse(error_body("rate_limited", f"Too many requests{limit}. {wait}"),
                                 status_code=429, headers=headers)
    if isinstance(exc, (litellm.exceptions.APIError, litellm.exceptions.ServiceUnavailableError,
                        litellm.exceptions.APIConnectionError, litellm.exceptions.Timeout,
                        litellm.exceptions.InternalServerError, openai.APIConnectionError, openai.InternalServerError)):
        ERROR_LOG.warning("provider error %s: %s", type(exc).__name__, str(exc)[:200])
        return 503, JSONResponse(error_body("provider_unavailable", "The model provider is unavailable right now. "
                                                                    "Try again in a minute."), status_code=503)
    ERROR_LOG.error("unhandled %s: %s", type(exc).__name__, str(exc)[:300])
    return 500, JSONResponse(error_body("internal_error", "Something went wrong on our side."), status_code=500)


def build_app(agent=None, store: Optional[PaperStore] = None) -> FastAPI:
    quiet = {"/health"}                    # health checks every few seconds: no spans, no log lines
    app = FastAPI(title="Paper explainer", version="4.0",
                  telemetry={"exclude": lambda scope: scope.get("path") in quiet})
    app.state.agents = {}
    app.state.agent = agent
    app.state.store = store or PaperStore(os.environ.get("DATA_DIR", "."))

    def agent_for(version: Optional[str]):
        if version is None or version == getattr(app.state.agent, "prompt_version", None):
            if app.state.agent is None:
                app.state.agent = build_agent()
            return app.state.agent
        if version not in ("v1", "v2"):
            raise HTTPException(422, "prompt_version must be v1 or v2.")
        if version not in app.state.agents:
            app.state.agents[version] = build_agent(version, extra_middleware=extra_middleware())
        return app.state.agents[version]

    def paper_or_404(paper_id: str) -> None:
        if not app.state.store.get(paper_id):
            raise HTTPException(404, f"No paper {paper_id}: upload it first (POST /papers).")

    @app.get("/health")
    def health():
        return {"status": "ok", "gateway_mode": config.GATEWAY_MODE}

    @app.post("/papers")
    def upload(request: Request, file: UploadFile = File(...), user: str = Depends(current_user)):  # noqa: B008
        data = file.file.read(config.MAX_UPLOAD_MB * 1024 * 1024 + 1)    # never read more than the limit + 1
        validate_upload(data)
        CURRENT_KEY.set(request.state.api_key)          # proxy mode: the uploader's key pays for the embeddings
        paper_id, is_new = app.state.store.ingest(data)
        chunks = len(app.state.store.get(paper_id)["chunks"])
        return {"paper_id": paper_id, "filename": file.filename, "chunks": chunks,
                "status": "processed" if is_new else "already processed"}

    @app.post("/papers/{paper_id}/ask")
    def ask(paper_id: str, body: Question, request: Request, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        result = ask_paper(paper_id, body.question, user, agent=agent_for(body.prompt_version),
                           api_key=request.state.api_key)
        request.state.usage = result
        return {"paper_id": paper_id, **result}

    @app.post("/papers/{paper_id}/explain")
    def explain(paper_id: str, request: Request, body: Optional[Question] = None, user: str = Depends(current_user)):
        paper_or_404(paper_id)
        events: queue.Queue = queue.Queue()
        agent = agent_for(body.prompt_version if body else None)

        def run():               # one thread for the whole run, so its spans open and close in one context
            try:
                for event, data in explain_events(paper_id, body.question if body else EXPLAIN_Q, user, agent,
                                                  request.state.api_key):
                    if event == "done":
                        request.state.usage = data
                    events.put((event, data))
            except Exception as exc:                       # mid-stream: the status is already 200
                status, payload = error_response(exc)
                events.put(("error", {**json.loads(payload.body), "status": status}))
            events.put(None)

        threading.Thread(target=run, daemon=True).start()

        def stream():
            while (item := events.get()) is not None:
                yield f"event: {item[0]}\ndata: {json.dumps(item[1])}\n\n"
        return StreamingResponse(stream(), media_type="text/event-stream")

    @app.get("/me/usage")
    def my_usage(request: Request, user: str = Depends(current_user)):
        if config.GATEWAY_MODE == "proxy":
            info = key_info(request.state.api_key)
            spent, budget = info.get("spend") or 0.0, info.get("max_budget")
            return {"user": user, "llm_calls": None, "spent_usd": round(spent, 6), "budget_usd": budget,
                    "remaining_usd": None if budget is None else round(budget - spent, 6), "source": "gateway"}
        return usage_of(user)

    # --- errors: a status and a sentence, never a stack trace ----------------------------------------
    @app.exception_handler(BudgetExceededError)
    @app.exception_handler(RateLimitedError)
    def too_much(request, exc):
        return error_response(exc)[1]

    @app.exception_handler(HTTPException)
    def http_error(request, exc: HTTPException):
        slug = {401: "unauthorized", 404: "not_found", 413: "too_large", 415: "unsupported_type",
                422: "invalid_request", 503: "gateway_unavailable"}.get(exc.status_code, "error")
        return JSONResponse(error_body(slug, exc.detail), status_code=exc.status_code)

    @app.middleware("http")
    async def request_log(request: Request, call_next):
        """One JSON line per request, written when the response has been sent. The question, the answer and
        the paper's text are never logged."""
        request_id = uuid.uuid4().hex[:8]
        started = time.perf_counter()
        try:
            response = await call_next(request)
        except Exception as exc:                            # anything not handled above
            status, response = error_response(exc)
        response.headers["X-Request-ID"] = request_id
        body = response.body_iterator if hasattr(response, "body_iterator") else None

        def write():
            usage = getattr(request.state, "usage", None) or {}
            LOG.info(json.dumps({
                "request_id": request_id, "user": getattr(request.state, "user", None),
                "route": f"{request.method} {request.url.path}", "status": response.status_code,
                "latency_ms": round((time.perf_counter() - started) * 1000),
                "llm_calls": usage.get("llm_calls", 0), "input_tokens": usage.get("input_tokens", 0),
                "output_tokens": usage.get("output_tokens", 0), "cost_usd": usage.get("cost_usd", 0.0),
                "answered_by": usage.get("answered_by", [])}))

        if request.url.path in quiet:
            return response
        if body is None:
            write()
            return response

        async def logged():
            async for chunk in body:
                yield chunk
            write()
        response.body_iterator = logged()
        return response

    return app


app = build_app(agent, papers)

def validate_upload(data: bytes) -> None:
    """415 unless the bytes start like a PDF; 413 if they are over MAX_UPLOAD_MB."""
    if not data.startswith(b"%PDF-"): raise HTTPException(415, "Only PDF files are accepted.")
    if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise HTTPException(413, f"Max upload size is {MAX_UPLOAD_MB} MB.")

def start_server(app: FastAPI, port: int = 8765) -> str:
    """4.4: serve `app` with uvicorn in a background thread of this notebook. Running it again stops the old
    server first, so the cell (and every catch-up cell) can be re-run."""
    import uvicorn
    if port in SERVERS:
        old, thread = SERVERS.pop(port)
        old.should_exit = True
        thread.join(timeout=10)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        if not thread.is_alive():
            raise RuntimeError(f"the server could not start: is port {port} in use? Runtime → Restart session, "
                               "then run 5.0")
        time.sleep(0.05)
    SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


BASE = start_server(app)
http = httpx.Client(base_url=BASE, timeout=120, trust_env=False)

def check_budget(user_id: str) -> None:
    """Raise BudgetExceededError once the user has spent their whole budget."""
    spent = SPEND[user_id]
    budget = BUDGETS.get(user_id, DEFAULT_BUDGET_USD)
    if spent >= budget:
        raise BudgetExceededError(user_id, spent, budget)


@before_model
def budget_guard(state, runtime):
    """Before EVERY model call, not once per question: one question can take several calls."""
    if config.GATEWAY_MODE == "proxy":
        return None
    check_budget(runtime.context.user_id)
    return None

agent = build_agent(router=router, extra_middleware=[budget_guard])    # 5.4: the guard joins the live agent
app.state.agent = agent
BUDGETS["alice"] = 1.00

log = attach_log_handler(LogCapture())      # prints each line, and keeps it for searching

exporter = setup_tracing(InMemorySpanExporter())      # from now on, every span is kept here
print('✅ caught up: ready for Section 7')

### 7.1 The golden set

Eight questions about TinyCoder with known right answers: six the paper answers (with the section the answer lives in) and two it can't, where the right answer is *the paper doesn't cover that*.

In [ ]:
GOLDEN = golden_set("notebook")
CHUNKS = chunk_index(papers.get(PAPER_ID))          # chunk id -> section, page, text

In [ ]:
# 7.1
for q in GOLDEN:
    expected = ", ".join(q["expected_sections"]) if q["answerable"] else "refuse: the paper doesn't cover it"
    print(f"{q['id']}  {q['question']:60} → {expected}")

### 7.2 Scorers and the LLM judge

Rules first, checked exactly by a program. Then one thing rules can't check: is every claim supported by the sources? A second model judges that, with its own versioned prompt. Free tier and 429s in 7.3? Set `EVAL_PAUSE_S = 4` here and run 7.2 and 7.3 again.

In [ ]:
JUDGE_VERSION = "judge_v1"
REFUSAL = "The paper doesn't cover that."
HYPE = ("breakthrough", "revolutionary", "game-changing", "groundbreaking", "unprecedented", "better than all",
        "state-of-the-art", "best ever", "incredible", "amazing")
REFUSED = re.compile(r"\b(doesn'?t|does not|didn'?t|did not) (cover|say|mention|report|state)|\bnot (covered|mentioned|"
                     r"reported|stated) in the paper", re.I)


def plain(text: str) -> str:
    """The answer without citation markers like [p2-c1]."""
    return " ".join(CITATION.sub("", text or "").split())


def syllables(word: str) -> int:
    word = re.sub(r"[^a-z]", "", word.lower())
    if not word:
        return 0
    groups = len(re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith(("le", "ee")) and groups > 1:
        groups -= 1                                   # silent e: "make", "code"
    return max(1, groups)


def reading_ease(text: str) -> float:
    """Flesch reading ease, kept between 0 and 100: higher is easier. 60-70 is plain English; under 30 reads
    like a paper."""
    words = [w for w in re.findall(r"[A-Za-z][A-Za-z'-]*|\d[\d,.%@]*", text)]
    if not words:
        return 0.0
    sentences = max(1, len(re.findall(r"[.!?]+(\s|$)", text)))
    syl = sum(syllables(w) if w[0].isalpha() else 1 for w in words)
    return round(min(100.0, max(0.0, 206.835 - 1.015 * len(words) / sentences - 84.6 * syl / len(words))), 1)


def is_refusal(answer: str) -> bool:
    return bool(REFUSED.search(answer or ""))


def score(item: dict, result: dict, chunks: dict, faithful: Optional[bool] = None) -> dict:
    """One question's scores. `result` is ask_paper()'s (or the api's) reply; `chunks` maps chunk id ->
    {"section": section id, "page": n}. None = this metric doesn't apply to this question."""
    answerable, answer, cited = item["answerable"], result["answer"], result.get("citations") or []
    sections = {chunks[c]["section"] for c in cited if c in chunks}
    pages = {chunks[c]["page"] for c in cited if c in chunks}
    retrieved_pages = {chunks[c]["page"] for c in result.get("retrieved") or [] if c in chunks}
    return {
        "valid_citations": 1.0 if result.get("citations_valid", True) else 0.0,
        "cited": (1.0 if cited else 0.0) if answerable else None,
        "right_section": (1.0 if sections & set(item["expected_sections"]) else 0.0) if answerable else None,
        "faithful": None if faithful is None else (1.0 if faithful else 0.0),
        "refused_correctly": None if answerable else (1.0 if is_refusal(answer) else 0.0),
        "hype_free": 0.0 if any(h in answer.lower() for h in HYPE) else 1.0,
        "readability": reading_ease(plain(answer)) if answerable and not is_refusal(answer) else None,
        "recall_at_k": (1.0 if item.get("expected_page") in retrieved_pages else 0.0) if answerable else None,
        "page_accuracy": (1.0 if item.get("expected_page") in pages else 0.0) if answerable else None,
    }


def judge_messages(question: str, sources: str, answer: str) -> list:
    return [{"role": "system", "content": JUDGE_V1},
            {"role": "user", "content": f"Question: {question}\n\nSources:\n{sources}\n\nAnswer:\n{answer}"}]


def parse_verdict(reply: str) -> dict:
    """{"faithful": bool, "reason": str}. Anything we can't read counts as NOT faithful: a judge that fails
    must never make a bad answer look good."""
    match = re.search(r"\{.*\}", reply or "", re.S)
    try:
        verdict = json.loads(match.group(0)) if match else None
    except json.JSONDecodeError:
        verdict = None
    if not isinstance(verdict, dict) or not isinstance(verdict.get("faithful"), bool):
        return {"faithful": False, "reason": f"unparseable judge reply: {(reply or '')[:60]!r}"}
    return {"faithful": verdict["faithful"], "reason": str(verdict.get("reason", ""))[:200]}


def judge(question: str, sources: str, answer: str, complete: Callable) -> dict:
    """A second model checks that every claim is supported. complete(messages, max_tokens) -> reply text."""
    from paper_agent.service.config import JUDGE_MAX_TOKENS
    return parse_verdict(complete(judge_messages(question, sources, answer), JUDGE_MAX_TOKENS))


JUDGE_V1 = load_prompt("judge", "v1")
EVAL_PAUSE_S = 0          # ← 4 on a free tier that allows ~20 requests per minute


def complete(messages, max_tokens):
    """The judge's model call: through the same gateway, capped at max_tokens."""
    return router.completion(model=ALIAS, messages=messages, max_tokens=max_tokens, temperature=0).choices[0].message.content

In [ ]:
# 7.2
print(JUDGE_V1)
for text in ("TinyCoder is small. It runs fast. It only knows Python.",
             "TinyCoder uses sliding-window attention in every layer, with a 1,024-token window: each token "
             "attends only to the 1,024 tokens before it."):
    print(f"\nreadability {reading_ease(text):5.1f}  {text[:90]}")

### 7.3 Run it and save a baseline

All eight questions through the same agent, a tick or a cross each, then the averages go to `baseline.json` with the prompt version, the judge version and the model. A baseline isn't a target of 100%; valid citations is the exception: always 1.0.

In [ ]:
NOTEBOOK_METRICS = ("valid_citations", "cited", "right_section", "faithful", "refused_correctly", "hype_free",
                    "readability")


def average(rows: list, metrics=NOTEBOOK_METRICS) -> dict:
    out = {}
    for m in metrics:
        values = [r["scores"][m] for r in rows if r["scores"].get(m) is not None]
        out[m] = None if not values else round(sum(values) / len(values), 1 if m == "readability" else 2)
    return out


def run_eval(prompt_version: str, ask: Callable, complete: Callable, chunks: dict, golden: Optional[list] = None,
             metrics=NOTEBOOK_METRICS, pause_s: Optional[float] = None, show: bool = True) -> tuple:
    """Ask every golden question, score it, judge it. ask(question) -> ask_paper()-style dict;
    complete(messages, max_tokens) -> text, for the judge. Returns (averages, rows)."""
    golden = golden or golden_set()
    pause_s = EVAL_PAUSE_S if pause_s is None else pause_s
    rows = []
    for item in golden:
        result = ask(item["question"])
        sources = "\n\n".join(f"[{c}] {chunks[c]['text']}" for c in result.get("retrieved") or [] if c in chunks)
        verdict = judge(item["question"], sources or "(nothing was retrieved)", result["answer"], complete)
        scores = score(item, result, chunks, faithful=verdict["faithful"])
        failed = [m for m in metrics if scores.get(m) == 0.0]
        rows.append({"id": item["id"], "question": item["question"], "scores": scores, "judge": verdict,
                     "llm_calls": result.get("llm_calls", 0), "cost_usd": result.get("cost_usd", 0.0)})
        if show:
            mark = "✅" if not failed else "❌"
            print(f"{mark} {item['question'][:62]:62} {'failed: ' + ', '.join(failed) if failed else ''}")
        if pause_s:
            time.sleep(pause_s)
    return average(rows, metrics), rows

In [ ]:
# 7.3 (about 22 model calls)
eval_v1 = build_agent("v1", router=router, extra_middleware=[budget_guard])
metrics_v1, rows_v1 = run_eval("v1", lambda q: ask_paper(PAPER_ID, q, "eval-bot", eval_v1), complete, CHUNKS, GOLDEN)
BASELINE = baseline_record(metrics_v1, "v1", os.environ["LLM_MODEL"].split(",")[0])
Path("baseline.json").write_text(json.dumps(BASELINE, indent=2))
print("\nbaseline.json:", json.dumps(BASELINE, indent=1))

### 7.4 Your turn: the regression gate. ✏️ TODO 3 (two lines)

Answers wobble a little between runs, so a small drop is allowed: 0.10 for most metrics, 10 points for
readability, **zero** for valid citations. **TODO 3:** if the current value is more than the allowed amount
below the baseline, add a message to `failures`.

In [ ]:
TOLERANCE = {"valid_citations": 0.0, "readability": 10.0}


def regression_gate(current: dict, baseline: dict) -> list:
    """The metrics that got worse than the baseline by more than their tolerance; [] means pass."""
    failures = []
    for metric, base in baseline.items():
        if metric not in current or base is None or current[metric] is None:
            continue
        allowed = TOLERANCE.get(metric, 0.10)
        # ✏️ TODO 3 (two lines): if current[metric] is more than `allowed` below `base`,
        #   append f"{metric}: {base} -> {current[metric]}" to failures
        pass
    return failures

### 7.4a Test: offline

Small wobbles pass; real drops are caught; a fake citation is never allowed.

In [ ]:
# 7.4a test
base = {"valid_citations": 1.0, "cited": 1.0, "right_section": 0.83, "readability": 48.0}
wobble = {"valid_citations": 1.0, "cited": 0.95, "right_section": 0.75, "readability": 40.0}
drop = {"valid_citations": 0.99, "cited": 0.0, "right_section": 0.83, "readability": 30.0}
assert regression_gate(wobble, base) == [], f"TODO 3: wobbles inside the tolerance should pass, got {regression_gate(wobble, base)}"
caught = [f.split(":")[0] for f in regression_gate(drop, base)]
print("caught:", regression_gate(drop, base))
assert caught == ["valid_citations", "cited", "readability"], f"TODO 3: expected valid_citations, cited and readability, got {caught}"
print("✅ TODO 3 passed: wobbles pass, real drops and any fake citation are caught")

<details><summary>🆘 Stuck? Hint</summary>

Two lines, inside the loop:

```
if current[metric] < base - allowed:
    failures.append(f"{metric}: {base} -> {current[metric]}")
```
</details>

🆘 **Rescue cell:** tests failing, or you fell behind? Run the next cell, then carry on.

In [ ]:
# 🆘 RESCUE: the finished version. Running it is always safe.
TOLERANCE = {"valid_citations": 0.0, "readability": 10.0}


def regression_gate(current: dict, baseline: dict) -> list:
    """The metrics that got worse than the baseline by more than their tolerance; [] means pass."""
    failures = []
    for metric, base in baseline.items():
        if metric not in current or base is None or current[metric] is None:
            continue
        allowed = TOLERANCE.get(metric, 0.10)
        if current[metric] < base - allowed:
            failures.append(f"{metric}: {base} -> {current[metric]}")
    return failures

### 7.5 A harmless prompt change

A teammate: *"the answers look cluttered with those [p2-c1] markers, can we make it friendlier?"* Prompt v2 says:
be friendly and conversational, no brackets or reference markers. Everything else is the same. Run it against
the baseline.

In [ ]:
# 7.5 (about 22 model calls)
print("v2's rule 3:", [line for line in load_prompt("agent", "v2").splitlines() if line.startswith("3.")][0], "\n")
eval_v2 = build_agent("v2", router=router, extra_middleware=[budget_guard])
metrics_v2, rows_v2 = run_eval("v2", lambda q: ask_paper(PAPER_ID, q, "eval-bot", eval_v2), complete, CHUNKS, GOLDEN)
baseline = json.loads(Path("baseline.json").read_text())
print()
compare(baseline["metrics"], metrics_v2, ("v1", "v2"))
failures = regression_gate(metrics_v2, baseline["metrics"])
if failures:
    print("\n❌ BLOCKED: v2 can't ship. Worse than the baseline:", " · ".join(failures))
else:
    print("\n✅ v2 passes the gate: on this model, the answers kept their quality (or TODO 3 isn't solved: run 7.4a)")

---
# Go live

### C.1 The go-live checklist

Each item, and the cell where you did it today. Two say *extension*: the first things to add next.

In [ ]:
# C.1
CHECKLIST = {
    "Serve": [("Auth on every endpoint", "4.2"), ("Validate uploads by bytes and size", "4.3"),
              ("Hash: never process a paper twice", "4.4"), ("Stream long answers", "4.6"),
              ("Clear errors, not stack traces", "4.2, 4.3a")],
    "Control": [("One gateway, one retry layer", "5.1"), ("Fallback model", "5.2"),
                ("Budget checked before every call", "5.3, 5.4"), ("max_tokens on every call", "4.0, 5.4"),
                ("Per-user rate limits with Retry-After", "5.5"), ("Spend visible per user", "5.6")],
    "Observe": [("One JSON log line per request", "6.1"), ("Request id in every response", "4.5, 6.1"),
                ("Traces across every step", "6.2"), ("Tokens and cost on every LLM span", "6.2"),
                ("Alerts: spend, errors, latency", "extension")],
    "Evaluate": [("Golden set, including refusals", "7.1"), ("Rules first, then a judge", "7.2"),
                 ("Version prompts and judges", "7.2, 7.5"), ("Baseline + gate before every change", "7.3-7.5"),
                 ("Weekly eval on sampled real traffic", "extension")],
    "Still on from Days 1-3": [("Untrusted-text guard, citation check, step limit", "4.0")],
}
for group, items in CHECKLIST.items():
    print(f"\n{group}")
    for item, cell in items:
        print(f"  ☐ {item:48} {cell}")

### C.2 Production cheatsheet

**Serve**
- Auth on every endpoint ([4.2](#scrollTo=day4-codealong-009))
- Validate uploads by content and size ([4.3](#scrollTo=day4-codealong-012))
- Hash documents; never process one twice ([4.4](#scrollTo=day4-codealong-018))
- Stream long answers ([4.6](#scrollTo=day4-codealong-023))
- Turn every known error into a clear status and message ([4.2](#scrollTo=day4-codealong-009))

**Control**
- One gateway for every LLM call ([5.1](#scrollTo=day4-codealong-027))
- Retries, timeout, fallback in one layer ([5.1](#scrollTo=day4-codealong-027), [5.2](#scrollTo=day4-codealong-030))
- Per-user budget checked before every call ([5.3](#scrollTo=day4-codealong-032))
- `max_tokens` on every call (limits overshoot) ([5.4](#scrollTo=day4-codealong-038))
- Per-user rate limits with Retry-After ([5.5](#scrollTo=day4-codealong-041))
- Spend visible per user ([5.6](#scrollTo=day4-codealong-043))

**Observe**
- One JSON log line per request, with a request id ([6.1](#scrollTo=day4-codealong-047))
- Keep document text and personal data out of general logs ([6.1](#scrollTo=day4-codealong-047))
- Traces across parse, retrieve, LLM and guards ([6.2](#scrollTo=day4-codealong-050))
- Tokens and cost on every LLM span ([6.2](#scrollTo=day4-codealong-050))
- Alerts on spend, errors and latency

**Evaluate**
- Golden set, including questions to refuse ([7.1](#scrollTo=day4-codealong-057))
- Rules first, an LLM judge for faithfulness ([7.2](#scrollTo=day4-codealong-060))
- Version prompts and judge prompts ([7.2](#scrollTo=day4-codealong-060))
- Baseline + regression gate before every change ([7.3](#scrollTo=day4-codealong-063), [7.4](#scrollTo=day4-codealong-066))
- Re-run on sampled real traffic to catch drift

---
### Extensions (homework)

**E.1 A small front end.** The next cell writes `ui.py`, a Streamlit page that uploads a PDF and asks your
service questions. Run it on your own machine: `pip install streamlit`, start the service (`uvicorn` or the
Docker repo's `make up`), then `streamlit run ui.py`.

**E.2 Real traces in Jaeger.** Swap `InMemorySpanExporter()` for an OTLP exporter pointed at Jaeger, and look
at the same tree in a browser.

**E.3 The real LiteLLM proxy.** The workshop repo's Docker setup runs the proxy with Postgres and Redis: one
virtual key per user with `max_budget`, `budget_duration` and `rpm_limit` (`make up`, then `make keys`).

**E.4 Grow the golden set to 20.** Add questions from your own use of the service, including two it should
refuse, and re-run the gate. The repo's `make eval` runs a 20-question set the way CI would.

In [ ]:
%%writefile ui.py
# E.1: a Streamlit front end for the Day 4 service. Run it locally: pip install streamlit; streamlit run ui.py
import os

import httpx
import streamlit as st

API = os.environ.get("API_URL", "http://127.0.0.1:8765")
st.title("Paper explainer")
st.caption("TinyCoder is a FICTIONAL paper written for the workshop. Never upload confidential documents.")
key = st.text_input("API key", value="key-alice", type="password")
headers = {"Authorization": f"Bearer {key}"}
upload = st.file_uploader("Upload a PDF", type=["pdf"])
if upload is not None and "paper_id" not in st.session_state:
    r = httpx.post(f"{API}/papers", headers=headers, timeout=120,
                   files={"file": (upload.name, upload.getvalue(), "application/pdf")})
    if r.status_code == 200:
        st.session_state.paper_id = r.json()["paper_id"]
        st.success(f"{r.json()['status']}: {st.session_state.paper_id}")
    else:
        st.error(r.json().get("message", r.text))
question = st.text_input("Your question", placeholder="How much faster is it than the baseline?")
if question and "paper_id" in st.session_state:
    r = httpx.post(f"{API}/papers/{st.session_state.paper_id}/ask", headers=headers, timeout=120,
                   json={"question": question})
    if r.status_code == 200:
        body = r.json()
        st.write(body["answer"])
        st.caption(f"citations {body['citations']} · {body['llm_calls']} LLM calls · ${body['cost_usd']:.6f} · "
                   f"answered by {', '.join(body['answered_by'])} · request {r.headers['X-Request-ID']}")
    else:
        st.error(f"{r.status_code}: {r.json().get('message', r.text)}")
usage = httpx.get(f"{API}/me/usage", headers=headers, timeout=30)
if usage.status_code == 200:
    st.sidebar.json(usage.json())